<a href="https://colab.research.google.com/github/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran/blob/main/SI26-Week4/SI26_Week4_humna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unified Urdu OCR Pipeline — Data Expansion, Fine-Tuning, and Audit

This merges the Week 4 (fine-tuning) and Week 5 (dataset expansion) notebooks into one
pipeline, in three phases:

1. **Data Pipeline & Preprocessing** — expand the dataset (synthetic rendering + document
   extraction), validate that every row's image path actually resolves, and cache
   preprocessed tensors.
2. **Pretrained Model Fine-Tuning** — initialize TrOCR and fine-tune on the expanded,
   validated dataset.
3. **Code Audit & Optimization** — profile the pipeline, empirically check the batch
   size/LR choices, and confirm best-checkpoint selection worked as intended.

**One ordering note on Phase 3, worth stating up front:** mixed precision, the LR
schedule, gradient clipping, and best-checkpoint-by-validation-CER are implemented
*inside* Phase 2's training loop, not bolted on afterward in Phase 3. Building them in
from the start is the point of having audited this pipeline already — training once
without them and fixing it after would just reproduce the original problem (see the
recap in Phase 3). Phase 3 instead profiles the run Phase 2 already did, and adds two
forward-looking empirical checks (a batch-size probe and an LR sanity check) that a
literal "optimize after training" step can actually deliver honestly, since you can't
un-spend the compute from an already-finished run.

**Carried over from the last two audits, still true here:**
- This does **not** include scrapers for BBC Urdu, Jang, Akhbar-e-Jahan, or Rekhta —
  commercial/copyrighted platforms; see the Phase 1 data-expansion cells for what's used
  instead.
- `microsoft/trocr-base-printed`'s decoder is RoBERTa, pretrained only on English — it
  has never seen Urdu script. That's most of why this notebook needs many more epochs
  than a small dataset would normally call for, and why accuracy has a real ceiling
  worth watching honestly (Phase 3 has more on this).
- Every path this notebook writes is relative to `DATA_DIR`, consistently — the specific
  bug that silently dropped 58% of the original dataset (mixed path conventions) doesn't
  recur here as long as that stays true.

---

**Debugging pass, after the first full 20-epoch run:** accuracy was still 0%, but this
was a different failure than the original one -- worth being precise about, since the fix
is different too. CER improved from 2.51 (original broken run) to 1.365 (this run's best),
and training loss dropped from ~17 to ~1.5-1.8 with no plateau. Real learning is
happening. But test predictions were dominated by runaway repeated n-grams
("و�و�و�و�و", "ررررررررر") rather than clean-but-wrong text -- classic exposure bias
(teacher-forced training loss doesn't directly penalize this; free-running generation
compounds errors in a way training loss doesn't see). Exact-match accuracy stays at 0%
until predictions are essentially perfect, so it wasn't going to move regardless.

Four fixes applied below, all additive to what Phase 2/3 already had:
1. `repetition_penalty`/`no_repeat_ngram_size` on every `generate()` call -- directly
   targets the repetition-loop symptom.
2. `eos_token_id` set explicitly on the model config (previously relied on the
   checkpoint's own default).
3. `NUM_EPOCHS` 20 → 40 -- the loss chart shows no convergence yet.
4. Synthetic-image generation made idempotent and re-enabled -- 90 of 180
   `synthetic_v2` rows in `labels.csv` had no matching image on Drive (safe to
   leave this cell on now; it skips anything that already exists).

An optional alternative-checkpoint comparison is at the end of Phase 2, per the
"if the architecture is inadequate" ask -- worth knowing that multiple people
independently hit this same RoBERTa-decoder-doesn't-know-Urdu wall trying to fine-tune
TrOCR on Urdu/Arabic/Persian and converged on the same fix direction (swap the decoder's
tokenizer), which is corroborating evidence for the diagnosis, not just this notebook's
own theory.

---

**Dataset-expansion pass:** 263 rows -- fewer still once the mislabeled `books`/
`newspaper` page-scans are removed in the audit below -- isn't enough data to fine-tune
a 334M-parameter sequence model, and that's a separate ceiling from the decoder-vocabulary
one above, not the same one restated. Section 1.8 (new) downloads real, purpose-built
Urdu OCR line datasets -- UTRSet-Real and UTRSet-Synth from the UTRNet project
(ICDAR'23), both printed text, matching this project's printed-text focus -- and folds
them into the same `labels.csv` merge that 1.1-1.3 already feed, via the same
`image, text, category` schema. UPTI is wired in but off by default; IIITH is
deliberately left out (its own creators designate it test-only). This is additive
only -- Phase 2 already trains on whatever ends up in `labels.csv` regardless of source,
so nothing above needed to change for it to pick up the larger dataset.
---

**CER-instability audit pass (this pass):** epochs 2/4/6 of the current 40-epoch run
showed validation CER of 0.952 -> 1.212 -> 1.078 -- rising, not falling, which reads like
overfitting at a glance. It isn't: training loss over those same epochs went
2.958 -> 2.627 -> 2.241 -> 1.797 -> 1.447, falling steadily with no plateau. Overfitting
requires loss to stay low/flat while validation error rises; loss dropping this fast this
early is the opposite signature. This is the same exposure-bias pattern already diagnosed
in the debugging pass above (teacher-forced training loss doesn't penalize free-running
generation errors), just still visible a few epochs into the *next* run despite the
repetition-penalty/eos_token_id mitigations already in place -- expected, not a regression,
since those mitigations reduce the damage rather than eliminate the underlying cause.

Two additions, both diagnostic (no hyperparameters changed -- see the reasoning above for
why touching LR/epochs/regularization here would be fixing the wrong problem):
1. Validation checks now report a full character-level breakdown (insertions/deletions/
   substitutions, hypothesis-vs-reference length ratio) alongside CER, not just the bare
   number -- lets you tell a repetition/length-runaway failure apart from the model just
   being wrong character-for-character.
2. A loss-vs-CER dashboard cell (2.3b, right after the loss chart) pairs training loss
   with validation CER at every checkpoint and prints an explicit overfitting/healthy/
   noise diagnosis per checkpoint, instead of eyeballing two numbers.


## Setup

In [1]:
import os
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/hamnasz/Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran.git"
DRIVE_ROOT = "/content/drive/MyDrive/Urdu-OCR-Project"
os.makedirs(DRIVE_ROOT, exist_ok=True)
REPO_DIR = os.path.join(DRIVE_ROOT, "urdu-ocr-repo")

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

DATA_DIR = os.path.join(REPO_DIR, "SI26-Week1", "data")
LABELS_PATH = os.path.join(DATA_DIR, "labels.csv")
print("DATA_DIR:", DATA_DIR)
print("labels.csv exists:", os.path.isfile(LABELS_PATH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR: /content/drive/MyDrive/Urdu-OCR-Project/urdu-ocr-repo/SI26-Week1/data
labels.csv exists: True


*(Not in the handout.)* Connect to the Hugging Face Hub once, up front — both loading
the pretrained checkpoint (Phase 2) and pushing the fine-tuned result back (end of this
notebook) need it.

In [2]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")  # never print this variable
login(token=HF_TOKEN)
print("Logged in to Hugging Face Hub.")

Logged in to Hugging Face Hub.


In [3]:
!pip install transformers==4.57.6 "pillow<12" pandas sentencepiece protobuf jiwer matplotlib huggingface_hub pdfplumber pypdfium2 python-docx gdown --quiet
# NOTE: torch is intentionally left out of this upgrade -- Colab's preinstalled torch/
# torchvision pair is already matched; a bare `pip install torch` here risks pulling a
# mismatched torchvision and breaking image ops.
import pandas as pd
import numpy as np
print("Ready.")

Ready.


In [4]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.")
    print("Training will still run on CPU, just far slower (hours instead of minutes).")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


---
# Phase 1: Data Pipeline & Preprocessing

Expand the dataset, then validate and cache it. Three expansion sources from your own
material (1.1–1.3) plus a fourth pulling in public research datasets (1.8), a merge step
(1.4), then loading the tokenizer/image processor and building the cached, validated
`Dataset` (1.5–1.7) that Phase 2 trains on directly.

## 1.1 Synthetic Rendering

Renders text to line images using Pillow's raqm/HarfBuzz-backed shaping
(`direction="rtl", language="ur"`) — the same approach Week 1/Week 3 already use, so
output is consistent with the rest of the project. Two things beyond what Week 3 did:

- **Multiple fonts** (`FONTS` below — both already in the repo; add more paths here if
  you have other properly-licensed Urdu fonts, e.g. more Noto/SIL-OFL-licensed families).
- **Augmentation** — mild rotation, blur, gaussian noise, and a varied paper-tone
  background instead of pure white, so the model sees some of the variation a scan or
  photo would introduce, instead of only ever training on perfectly clean synthetic text.

`make_dataset()` is corpus-agnostic: pass it any DataFrame with a text column. It's used
below with a *small* sample of the existing headlines corpus to prove it works end-to-end
— see the note in the intro cell about why this stays small rather than "render
everything." If you need to actually scale this, swap in a clean-provenance corpus (Wikipedia
Urdu, or the OCR-specific datasets in the summary cell) as the `corpus_df` argument.

In [5]:
import random
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageOps

FONTS = {
    "nastaliq": os.path.join(REPO_DIR, "SI26-Week1", "NotoNastaliqUrdu-Regular.ttf"),
    "naskh": os.path.join(REPO_DIR, "SI26-Week3", "fonts", "NotoNaskhArabic-Variable.ttf"),
}


def render_line(text, font_path, font_size=44, padding=18):
    """Render one line of Urdu text to a clean image (raqm-backed RTL shaping)."""
    font = ImageFont.truetype(font_path, font_size)
    tmp = Image.new("L", (10, 10))
    d = ImageDraw.Draw(tmp)
    bbox = d.textbbox((0, 0), text, font=font, direction="rtl", language="ur")
    w = int(bbox[2] - bbox[0]) + 2 * padding
    h = int(bbox[3] - bbox[1]) + 2 * padding
    img = Image.new("RGB", (w, h), "white")
    draw = ImageDraw.Draw(img)
    draw.text((padding - bbox[0], padding - bbox[1]), text, font=font, fill="black",
               direction="rtl", language="ur")
    return img


def augment(img, rng):
    """Mild randomized degradations: paper-tone background, slight rotation, blur, noise."""
    tint = rng.randint(235, 255)
    bg = Image.new("RGB", img.size, (tint, tint - rng.randint(0, 8), tint - rng.randint(0, 15)))
    mask = ImageOps.invert(img.convert("L")).point(lambda p: min(255, int(p * 1.15)))
    bg.paste((20, 20, 20), (0, 0), mask)
    img = bg

    angle = rng.uniform(-1.5, 1.5)
    img = img.rotate(angle, expand=True, fillcolor=(tint, tint, tint), resample=Image.BICUBIC)

    if rng.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.3, 0.9)))

    if rng.random() < 0.6:
        arr = np.array(img).astype(np.int16)
        noise = np.random.default_rng(rng.randint(0, 2**31)).normal(0, rng.uniform(3, 10), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)

    return img


def make_dataset(corpus_df, text_col, out_dir, data_dir, n_per_line=2, category="synthetic_v2", seed=0):
    """Render each row n_per_line times (random font + augmentation each time). Paths in
    the returned DataFrame are relative to data_dir -- the convention the Week 4 audit
    fixed labels.csv to use consistently; keep writing rows this way so nothing regresses.

    Idempotent (NEW): skips rendering any file that already exists on disk instead of
    always re-rendering. Safe to leave this cell enabled on every re-run -- previously
    rendered images (e.g. persisted on Drive from an earlier session) are left alone;
    only genuinely missing ones get (re)generated. This is what fixes the gap where
    labels.csv remembers rows whose image file never actually made it to Drive.
    """
    rng = random.Random(seed)
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    n_skipped = n_rendered = 0
    for i, text in enumerate(corpus_df[text_col].astype(str)):
        text = text.strip()
        if not (3 <= len(text) <= 90):
            continue
        for k in range(n_per_line):
            fname = f"{category}_{i:06d}_{k}.png"
            fpath = os.path.join(out_dir, fname)
            if os.path.isfile(fpath):
                n_skipped += 1
            else:
                font_name = rng.choice(list(FONTS.keys()))
                font_size = rng.randint(36, 52)
                img = augment(render_line(text, FONTS[font_name], font_size=font_size), rng)
                img.convert("RGB").save(fpath)
                n_rendered += 1
            rows.append({"image": os.path.relpath(fpath, data_dir), "text": text, "category": category})
    print(f"  make_dataset({category}): {n_rendered} rendered, {n_skipped} already existed and were left alone")
    return pd.DataFrame(rows, columns=["image", "text", "category"])

In [6]:
corpus_source = pd.read_csv(os.path.join(REPO_DIR, "SI26-Week3", "corpus", "headlines.csv"), sep="\t")
sample = corpus_source[corpus_source["title"].astype(str).str.len().between(10, 60)].sample(30, random_state=0)

synthetic_v2_out = os.path.join(DATA_DIR, "raw", "synthetic_v2")
df_synthetic_v2 = make_dataset(sample, "title", synthetic_v2_out, DATA_DIR, n_per_line=3, category="synthetic_v2")
print(f"df_synthetic_v2 covers {len(df_synthetic_v2)} rows ({sample.shape[0]} lines x up to 3 variants each) -- "
      f"re-enabled and now idempotent (see make_dataset above), so this is safe to leave on.")
df_synthetic_v2.head()

  make_dataset(synthetic_v2): 0 rendered, 90 already existed and were left alone
df_synthetic_v2 covers 90 rows (30 lines x up to 3 variants each) -- re-enabled and now idempotent (see make_dataset above), so this is safe to leave on.


,image,text,category
0,raw/synthetic_v2/synthetic_v2_000000_0.png,فیصل آباد اور مچھ میں 2 مجرموں کو پھانسی دے د...,synthetic_v2
1,raw/synthetic_v2/synthetic_v2_000000_1.png,فیصل آباد اور مچھ میں 2 مجرموں کو پھانسی دے د...,synthetic_v2
2,raw/synthetic_v2/synthetic_v2_000000_2.png,فیصل آباد اور مچھ میں 2 مجرموں کو پھانسی دے د...,synthetic_v2
3,raw/synthetic_v2/synthetic_v2_000001_0.png,ٹی ٹونٹی اورون ڈے سیریز کےلئے 9 کھلاڑی جنوبی ...,synthetic_v2
4,raw/synthetic_v2/synthetic_v2_000001_1.png,ٹی ٹونٹی اورون ڈے سیریز کےلئے 9 کھلاڑی جنوبی ...,synthetic_v2


## 1.2 PDF Extraction (text-layer PDFs)

For PDFs you have rights to use — your own scans, open textbooks, government
publications, course materials — that have a **real text layer** (not just a scanned
image). This crops the actual rendered page to each line's position and pairs it with the
exact extracted text: real fonts, kerning, and layout, which is a different (and useful)
kind of diversity than synthetic rendering.

Uses `pypdfium2` (Apache/BSD) to render pages and `pdfplumber` (MIT) to read line
positions — both permissively licensed, per the guidance in this environment's PDF
skill, which also flags PyMuPDF/`fitz` as AGPL-3.0 (copyleft) if you're deciding between
libraries elsewhere.

**Drop your own PDFs into `DATA_DIR/user_docs/pdf/` before running the cell below.**
Pages with no text layer (scanned images) are detected and skipped, not silently
mishandled — see the OCR-assist fallback cell further down if you want to pull
unverified "silver" labels from those instead (explicitly not ground truth).

**One caveat worth knowing:** word order within a line depends on how the source PDF
encoded its glyphs, which varies by whatever tool created it. This worked correctly on
every PDF tested while building this, but spot-check a handful of outputs against the
source document before trusting a new PDF source at scale.

In [7]:
import pdfplumber
import pypdfium2 as pdfium


def extract_pdf_lines(pdf_path, out_dir, data_dir, category="pdf_real", render_scale=3.0, min_chars=3):
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    scanned_pages = []
    base = os.path.splitext(os.path.basename(pdf_path))[0]

    pdf_render = pdfium.PdfDocument(pdf_path)
    with pdfplumber.open(pdf_path) as pdf_text:
        for page_idx, page_text in enumerate(pdf_text.pages):
            text_lines = page_text.extract_text_lines(strip=True, return_chars=False)
            if not text_lines:
                scanned_pages.append(page_idx)
                continue

            page_img = pdf_render[page_idx].render(scale=render_scale).to_pil()
            for li, line in enumerate(text_lines):
                text = line["text"].strip()
                if len(text) < min_chars:
                    continue
                pad = 4
                box = (
                    max(0, line["x0"] * render_scale - pad),
                    max(0, line["top"] * render_scale - pad),
                    min(page_img.width, line["x1"] * render_scale + pad),
                    min(page_img.height, line["bottom"] * render_scale + pad),
                )
                if box[2] - box[0] < 5 or box[3] - box[1] < 5:
                    continue
                crop = page_img.crop(box).convert("RGB")
                fname = f"{category}_{base}_{page_idx:03d}_{li:03d}.png"
                fpath = os.path.join(out_dir, fname)
                crop.save(fpath)
                rows.append({"image": os.path.relpath(fpath, data_dir), "text": text, "category": category})

    if scanned_pages:
        print(f"  {pdf_path}: {len(scanned_pages)} page(s) had no text layer (scanned?) -- "
              f"skipped: {scanned_pages}. See the OCR-assist fallback cell for that case.")
    return pd.DataFrame(rows, columns=["image", "text", "category"])


def extract_pdf_scanned_silver(pdf_path, lang="urd", render_scale=3.0):
    """OCR-assisted 'silver' labels for scanned pages, via Tesseract's Urdu model
    (`apt-get install tesseract-ocr-urd` if not already present). NOT ground truth --
    bootstrapping labels from another OCR system's output is circular. Rows come back
    tagged category='silver_needs_review' so they're easy to route to human review or
    filter out entirely before ever training on them.
    """
    import pytesseract
    pdf_render = pdfium.PdfDocument(pdf_path)
    rows = []
    for page_idx, page in enumerate(pdf_render):
        img = page.render(scale=render_scale).to_pil()
        text = pytesseract.image_to_string(img, lang=lang).strip()
        if text:
            rows.append({"page": page_idx, "text": text, "category": "silver_needs_review"})
    return pd.DataFrame(rows)

In [8]:
import glob

pdf_input_dir = os.path.join(DATA_DIR, "user_docs", "pdf")
os.makedirs(pdf_input_dir, exist_ok=True)
# pdf_files = glob.glob(os.path.join(pdf_input_dir, "*.pdf"))
pdf_files = [] # Commented out to prevent loading PDF files
print(f"Found {len(pdf_files)} PDF(s) in {pdf_input_dir}")

pdf_out = os.path.join(DATA_DIR, "raw", "pdf_v2")
# df_pdf_list = [extract_pdf_lines(p, pdf_out, DATA_DIR) for p in pdf_files]
# df_pdf = pd.concat(df_pdf_list, ignore_index=True) if df_pdf_list else pd.DataFrame(columns=["image", "text", "category"])
df_pdf = pd.DataFrame(columns=["image", "text", "category"]) # Initialize as empty DataFrame
print(f"Extracted {len(df_pdf)} line images from {len(pdf_files)} PDF(s)")
df_pdf.head()

Found 0 PDF(s) in /content/drive/MyDrive/Urdu-OCR-Project/urdu-ocr-repo/SI26-Week1/data/user_docs/pdf
Extracted 0 line images from 0 PDF(s)


,image,text,category


## 1.3 DOCX Extraction

Pulls paragraph text out of your own `.docx` files. DOCX is a flow document with no fixed
page layout, so there's no bounding box to crop an image from — the extracted text gets
fed through Step 1's `render_line()`/`augment()` instead, same as any other text corpus.

**Drop your own `.docx` files into `DATA_DIR/user_docs/docx/` before running.**

In [9]:
from docx import Document


def chunk_paragraph(text, max_chars=90):
    """Split a long paragraph into line-length pieces on word boundaries."""
    words = text.split()
    chunks, cur = [], ""
    for w in words:
        if cur and len(cur) + len(w) + 1 > max_chars:
            chunks.append(cur.strip())
            cur = w
        else:
            cur = (cur + " " + w).strip()
    if cur:
        chunks.append(cur.strip())
    return chunks


def extract_docx_lines(docx_path, max_chars=90, min_chars=3):
    doc = Document(docx_path)
    lines_out = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if text:
            lines_out.extend(chunk_paragraph(text, max_chars=max_chars))
    return [l for l in lines_out if len(l) >= min_chars]

In [10]:
docx_input_dir = os.path.join(DATA_DIR, "user_docs", "docx")
os.makedirs(docx_input_dir, exist_ok=True)
# docx_files = glob.glob(os.path.join(docx_input_dir, "*.docx"))
docx_files = [] # Commented out to prevent loading DOCX files
print(f"Found {len(docx_files)} DOCX file(s) in {docx_input_dir}")

# all_docx_lines = []
# for p in docx_files:
#     all_docx_lines.extend(extract_docx_lines(p))
all_docx_lines = [] # Commented out to prevent DOCX line extraction
print(f"Extracted {len(all_docx_lines)} line-length chunks")

docx_out = os.path.join(DATA_DIR, "raw", "docx_v2")
# if all_docx_lines:
#     df_docx_corpus = pd.DataFrame({"text": all_docx_lines})
#     df_docx = make_dataset(df_docx_corpus, "text", docx_out, DATA_DIR, n_per_line=1, category="docx_v2")
# else:
#     df_docx = pd.DataFrame(columns=["image", "text", "category"])
df_docx = pd.DataFrame(columns=["image", "text", "category"]) # Initialize as empty DataFrame
print(f"Rendered {len(df_docx)} images from DOCX text")
df_docx.head()

Found 0 DOCX file(s) in /content/drive/MyDrive/Urdu-OCR-Project/urdu-ocr-repo/SI26-Week1/data/user_docs/docx
Extracted 0 line-length chunks
Rendered 0 images from DOCX text


,image,text,category


### 1.8 Public Urdu OCR Datasets (External Sources)

*(Not in the handout -- added because 263 rows, and fewer still once the mislabeled
`books`/`newspaper` page-scans are removed below, isn't enough data to fine-tune a
334M-parameter sequence model. That's a separate problem from the RoBERTa-decoder one in
the intro cell -- more data doesn't fix a decoder that's never seen an Urdu subword, but
it's necessary regardless of which checkpoint ends up being used.)*

Downloads real, purpose-built Urdu OCR **line** datasets and folds them into the same
`image, text, category` schema every other Phase 1 source uses, so they merge into
`labels.csv` in 1.4 exactly like `df_synthetic_v2`/`df_pdf`/`df_docx` do.

**Sources (all released as public Google Drive files by the UTRNet/ICDAR'23 authors --
see [the project's GitHub repo](https://github.com/abdur75648/UTRNet-High-Resolution-Urdu-Text-Recognition#datasets)):**

| Source | What it is | ~Rows | License |
|---|---|---|---|
| `UTRSet-Real` | Manually annotated, real-world printed Urdu lines | 11,000 | CC BY-NC-4.0 |
| `UTRSet-Synth` | Synthetic printed Urdu lines, same paper | 20,000 | CC BY-NC-4.0 |
| `UPTI` | CLE Lahore's synthetic printed Nastaliq lines -- **off by default** | 10,000 | Research use (see source paper) |

`UTRSet-Real`/`UTRSet-Synth` are both **printed** text -- matching the synthetic
rendering in 1.1 and the `microsoft/trocr-base-printed` checkpoint in Phase 2, so there's
no handwritten/printed domain mismatch stacked on top of everything else going on.

**Deliberately left out:** the "IIITH (Updated)" release on the same page is a corrected
re-release of IIIT-Delhi's IIIT-Urdu OCR set, which its original creators
[designate test-only](https://ilocr.iiit.ac.in/dataset/38/) -- folding it into training
data would contaminate any evaluation done against it elsewhere. `UPTI` is wired in below
but off by default since its license terms are less explicit than UTRSet's -- flip
`enabled` to `True` in `PUBLIC_DATASET_SOURCES` once you've checked it fits your use.

**How this behaves at runtime:**
- Downloads via `gdown` to **local Colab disk**, not straight to Drive -- Drive-mounted
  writes are slow for lots of small files, and there's no reason to spend Drive quota on
  a temporary zip. Only the final set of line images (capped, resized) gets copied into
  `DATA_DIR/raw/<category>/`, which *is* on Drive, same as every other source here.
- **Google Drive enforces a daily download quota on popular shared files** -- a `gdown`
  failure here is most often that, not a broken link. Each source is wrapped in its own
  try/except: a failed source prints a clear message (with a manual-download fallback)
  and is skipped, rather than crashing the rest of Phase 1.
- The ground-truth format (an `images/` folder plus one or more `*_gt.txt` files,
  `image_path<TAB>text` per line -- the convention this whole dataset family uses) is
  auto-detected rather than hardcoded to one exact folder layout, since it isn't
  byte-identical across all of UTRNet's releases.
- `max_samples` defaults to 3,000 per source below -- both defaults enabled already takes
  `labels.csv` from ~263 rows to 6,000+, a ~20x increase, without a first run accidentally
  taking hours on a free-tier GPU. Set it to `None` for "use everything," in which case
  you'll likely also want to lower `NUM_EPOCHS` in Phase 2 -- that tradeoff is a training
  decision, so it isn't changed automatically here.

In [11]:
import zipfile
import gdown

IMG_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
LOCAL_DOWNLOAD_DIR = "/content/public_urdu_datasets"  # local disk -- fast, no Drive quota spent on zips

PUBLIC_DATASET_SOURCES = {
    "utrset_real": {
        "enabled": True,
        "gdrive_id": "1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T",
        "category": "public_utrset_real",
        "description": "UTRSet-Real -- real-world printed Urdu lines (UTRNet, ICDAR'23)",
        "license": "CC BY-NC-4.0, non-commercial research",
        "max_samples": 3000,
    },
    "utrset_synth": {
        "enabled": True,
        "gdrive_id": "18hN2Ab2XtjiRJigogOd2DUBKa0zV0qDL",
        "category": "public_utrset_synth",
        "description": "UTRSet-Synth -- synthetic printed Urdu lines (UTRNet, ICDAR'23)",
        "license": "CC BY-NC-4.0, non-commercial research",
        "max_samples": 3000,
    },
    "upti": {
        "enabled": False,  # off by default -- check the license fits your use before enabling
        "gdrive_id": "1lTrisSd_ZwlVsMXqVdpMD6hMOFSWWHru",
        "category": "public_upti",
        "description": "UPTI -- CLE Lahore's synthetic printed Nastaliq lines",
        "license": "Research use -- see https://ui.adsabs.harvard.edu/abs/2013SPIE.8658E..0NS",
        "max_samples": 3000,
    },
}


def download_and_extract_gdrive_zip(gdrive_id, dest_dir):
    """Download a Google-Drive-hosted zip via gdown and extract it to local disk.
    Returns the extraction directory, or None if the download failed -- most often
    Google Drive's daily quota on a popular shared file, not a broken link. Idempotent:
    a re-run with the extraction already present skips straight past the download."""
    os.makedirs(dest_dir, exist_ok=True)
    extract_dir = os.path.join(dest_dir, gdrive_id)
    if os.path.isdir(extract_dir) and os.listdir(extract_dir):
        print(f"  already extracted at {extract_dir} -- skipping download")
        return extract_dir
    zip_path = os.path.join(dest_dir, f"{gdrive_id}.zip")
    try:
        if not os.path.isfile(zip_path):
            gdown.download(id=gdrive_id, output=zip_path, quiet=False)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        try:
            os.remove(zip_path)  # keep local disk usage down once unzipped
        except OSError:
            pass
        return extract_dir
    except Exception as e:
        print(f"  DOWNLOAD/EXTRACT FAILED ({e.__class__.__name__}: {e})")
        print(f"  Usually Google Drive's daily quota on a popular shared file -- try again "
              f"later, or download by hand from https://drive.google.com/file/d/{gdrive_id}/view "
              f"and unzip it to {extract_dir}")
        if os.path.isfile(zip_path):
            try:
                os.remove(zip_path)  # don't leave a corrupt partial download for the next re-run to trip on
            except OSError:
                pass
        return None


def _find_gt_files(root_dir):
    """Find *_gt.txt-style files: any .txt whose first non-empty line is exactly two
    tab-separated fields where the first looks like an image path. This is the
    convention IIIT-Urdu/UTRNet-family releases use (`images/name.png<TAB>text`),
    auto-detected rather than hardcoded since the exact filename/folder isn't identical
    across all of them."""
    candidates = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fn in filenames:
            if not fn.lower().endswith(".txt"):
                continue
            fpath = os.path.join(dirpath, fn)
            try:
                with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
                    for line in f:
                        if not line.strip():
                            continue
                        parts = line.rstrip("\n").split("\t")
                        if len(parts) == 2 and parts[0].strip().lower().endswith(IMG_EXTENSIONS):
                            candidates.append(fpath)
                        break
            except Exception:
                continue
    return candidates


def _resolve_image_path(root_dir, gt_dir, rel_path):
    """gt.txt image paths are usually relative to the gt file's own folder, but try a
    couple of other reasonable resolutions too before giving up on that row."""
    for c in (os.path.join(gt_dir, rel_path), os.path.join(root_dir, rel_path),
              os.path.join(gt_dir, os.path.basename(rel_path))):
        if os.path.isfile(c):
            return c
    return None


def ingest_gt_style_dataset(extracted_dir, category, out_dir, data_dir, max_samples=None,
                             max_height=140, jpeg_quality=90, seed=0):
    """Normalize an images/ + *_gt.txt release into this project's (image, text,
    category) schema -- same convention make_dataset()/extract_pdf_lines() already use,
    so it merges into labels.csv identically. Images are re-saved as JPEG capped at
    max_height (matching typical line-image scale and keeping Drive usage down) rather
    than copied at whatever resolution the source used."""
    gt_files = _find_gt_files(extracted_dir)
    if not gt_files:
        print(f"  [{category}] no recognizable *_gt.txt found under {extracted_dir} -- skipping")
        return pd.DataFrame(columns=["image", "text", "category"])

    os.makedirs(out_dir, exist_ok=True)
    rng = random.Random(seed)
    pairs = []
    for gt_path in gt_files:
        gt_dir = os.path.dirname(gt_path)
        with open(gt_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                parts = line.rstrip("\n").split("\t", 1)
                if len(parts) != 2 or not parts[1].strip():
                    continue
                pairs.append((gt_dir, parts[0].strip(), parts[1].strip()))

    print(f"  [{category}] {len(pairs)} labeled line(s) across {len(gt_files)} gt file(s)")
    if max_samples and len(pairs) > max_samples:
        rng.shuffle(pairs)
        pairs = pairs[:max_samples]
        print(f"  [{category}] capped to {max_samples} (set max_samples=None in "
              f"PUBLIC_DATASET_SOURCES to use all)")

    rows, n_missing = [], 0
    for i, (gt_dir, rel_img, text) in enumerate(pairs):
        src = _resolve_image_path(extracted_dir, gt_dir, rel_img)
        if src is None:
            n_missing += 1
            continue
        fname = f"{category}_{i:06d}.jpg"
        fpath = os.path.join(out_dir, fname)
        if not os.path.isfile(fpath):
            try:
                img = Image.open(src).convert("RGB")
                if img.height > max_height:
                    ratio = max_height / img.height
                    img = img.resize((max(1, int(img.width * ratio)), max_height), Image.LANCZOS)
                img.save(fpath, "JPEG", quality=jpeg_quality)
            except Exception:
                n_missing += 1
                continue
        rows.append({"image": os.path.relpath(fpath, data_dir), "text": text, "category": category})

    if n_missing:
        print(f"  [{category}] {n_missing} row(s) had an image that couldn't be found/opened -- skipped")
    print(f"  [{category}]: {len(rows)} rows ready")
    return pd.DataFrame(rows, columns=["image", "text", "category"])

In [12]:
df_public_list = []
for name, cfg in PUBLIC_DATASET_SOURCES.items():
    print(f"\n=== {name}: {cfg['description']} ({cfg['license']}) ===")
    if not cfg["enabled"]:
        print(f"  disabled -- set PUBLIC_DATASET_SOURCES['{name}']['enabled'] = True to include it")
        continue
    extracted = download_and_extract_gdrive_zip(cfg["gdrive_id"], LOCAL_DOWNLOAD_DIR)
    if extracted is None:
        continue
    out_dir = os.path.join(DATA_DIR, "raw", cfg["category"])
    df_source = ingest_gt_style_dataset(extracted, cfg["category"], out_dir, DATA_DIR,
                                         max_samples=cfg.get("max_samples"))
    df_public_list.append(df_source)

df_public = (pd.concat(df_public_list, ignore_index=True) if df_public_list
             else pd.DataFrame(columns=["image", "text", "category"]))
print(f"\ndf_public covers {len(df_public)} rows total from public external sources")
df_public.head()


=== utrset_real: UTRSet-Real -- real-world printed Urdu lines (UTRNet, ICDAR'23) (CC BY-NC-4.0, non-commercial research) ===
  already extracted at /content/public_urdu_datasets/1mABkzaWe1hLikXCaM5nmLsBCWtxvDE7T -- skipping download
  [public_utrset_real] 11131 labeled line(s) across 2 gt file(s)
  [public_utrset_real] capped to 3000 (set max_samples=None in PUBLIC_DATASET_SOURCES to use all)
  [public_utrset_real]: 3000 rows ready

=== utrset_synth: UTRSet-Synth -- synthetic printed Urdu lines (UTRNet, ICDAR'23) (CC BY-NC-4.0, non-commercial research) ===
  already extracted at /content/public_urdu_datasets/18hN2Ab2XtjiRJigogOd2DUBKa0zV0qDL -- skipping download
  [public_utrset_synth] no recognizable *_gt.txt found under /content/public_urdu_datasets/18hN2Ab2XtjiRJigogOd2DUBKa0zV0qDL -- skipping

=== upti: UPTI -- CLE Lahore's synthetic printed Nastaliq lines (Research use -- see https://ui.adsabs.harvard.edu/abs/2013SPIE.8658E..0NS) ===
  disabled -- set PUBLIC_DATASET_SOURCES['upti

,image,text,category
0,raw/public_utrset_real/public_utrset_real_0000...,فارسی,public_utrset_real
1,raw/public_utrset_real/public_utrset_real_0000...,(۱۱),public_utrset_real
2,raw/public_utrset_real/public_utrset_real_0000...,شوکت داراحشمت سکندر مرتبت شمس الصحٰی بدرالدجیٰ...,public_utrset_real
3,raw/public_utrset_real/public_utrset_real_0000...,ہی ہاتھوں دفن کردیناپڑتاہے---کیایہ ظلم نہیں ہے...,public_utrset_real
4,raw/public_utrset_real/public_utrset_real_0000...,ساتھی چن لیاتھا! شروع ہی سے وہ خاندانوں میں سب...,public_utrset_real


## 1.4 Merge Expansions Into `labels.csv`

Backs up the current `labels.csv` first, then appends whatever new rows exist from Steps
1–3. Safe to re-run — rerunning Steps 1–3 with the same inputs regenerates the same
filenames, and this step only adds rows for images that exist and aren't already listed.

In [13]:
import shutil, datetime

backup_path = LABELS_PATH + f".backup-{datetime.datetime.now():%Y%m%d-%H%M%S}"
shutil.copy(LABELS_PATH, backup_path)
print("Backed up existing labels.csv to:", backup_path)

existing = pd.read_csv(LABELS_PATH)
new_rows = pd.concat([df_synthetic_v2, df_pdf, df_docx, df_public], ignore_index=True)
new_rows = new_rows[~new_rows["image"].isin(existing["image"])]  # skip anything already listed

combined = pd.concat([existing, new_rows], ignore_index=True)
combined.to_csv(LABELS_PATH, index=False)

print(f"labels.csv: {len(existing)} existing rows + {len(new_rows)} new rows = {len(combined)} total")
print(combined["category"].value_counts(dropna=False))

Backed up existing labels.csv to: /content/drive/MyDrive/Urdu-OCR-Project/urdu-ocr-repo/SI26-Week1/data/labels.csv.backup-20260807-175151
labels.csv: 3280 existing rows + 0 new rows = 3280 total
category
public_utrset_real    3000
synthetic              110
synthetic_v2            90
NaN                     80
Name: count, dtype: int64


## 1.5 Load the Tokenizer/Image Processor

Preprocessing (tokenizing text, resizing/normalizing images) needs the same
tokenizer/image processor the pretrained checkpoint was trained with, so that part loads
now, in Phase 1 — separately from the model weights themselves, which load in Phase 2
under "initialize the pretrained checkpoint." Same artifact either way
(`TrOCRProcessor`), just split by what each phase actually needs it for.

In [14]:
from transformers import TrOCRProcessor

# Handout text has this as 'microsoft/trocr-baseprinted' (missing hyphen) -- that's not
# a real model on the Hub. The correct id (matching Week 3 and the handout's own Sources
# section) is set here as MODEL_NAME so Phase 2 loads the matching weights below.
MODEL_NAME = "microsoft/trocr-base-printed"

try:
    processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
except Exception as e:
    raise RuntimeError(
        f"Couldn't load the processor for '{MODEL_NAME}' from Hugging Face Hub: {e}. "
        "Check your internet connection and that huggingface.co is reachable."
    ) from e

print("Processor loaded (tokenizer + image processor).")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Processor loaded (tokenizer + image processor).


## 1.6 Right-Size `MAX_LENGTH`

*(Not in the handout.)* The handout hardcodes `MAX_LENGTH = 128` as a guess. This measures
it instead: every label in `labels.csv` gets tokenized with the real TrOCR tokenizer, and
`MAX_LENGTH` is set to the longest sequence actually found (plus a small buffer for the
special tokens `generate()` adds), never below the handout's original 128.

This matters more here than it might elsewhere: Urdu isn't in this tokenizer's training
data (previous cell), so it falls back to encoding raw UTF-8 bytes rather than efficient
learned subwords, and Urdu characters take ~1.8 UTF-8 bytes each on average versus close to
1 for English. That inflates token counts well past what 128 was likely sized for, which
means labels were silently getting truncated during training on top of everything else.

In [15]:
import pandas as pd

_texts = pd.read_csv(LABELS_PATH)["text"].astype(str).tolist()
_lengths = [len(processor.tokenizer(t).input_ids) for t in _texts]

print(f"Tokenized length over {len(_texts)} labels -- "
      f"min: {min(_lengths)}, mean: {sum(_lengths) / len(_lengths):.1f}, max: {max(_lengths)}")
print(f"Rows that would truncate at the handout's MAX_LENGTH=128: {sum(l > 128 for l in _lengths)}")

# Longest real sequence + a small buffer for decoder_start/eos, never below the original 128.
MAX_LENGTH = max(128, max(_lengths) + 8)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

Tokenized length over 3280 labels -- min: 3, mean: 68.7, max: 186
Rows that would truncate at the handout's MAX_LENGTH=128: 114
Using MAX_LENGTH = 194


## 1.7 Validate Paths and Cache Preprocessed Tensors

Two things happen in the `UrduOCRDataset` class below, matching the two asks for this
phase:

**Validate paths/labels** — `resolve_image_path()` checks each row's image against both
path conventions this repo's `labels.csv` has accumulated (see the overview cell), and
rows that resolve to neither are dropped with a printed count rather than silently
producing a `FileNotFoundError` mid-training. The validation report cell right after this
one makes the resolution outcome explicit rather than leaving it implicit in a print
statement.

**Cache preprocessed tensors** — every image's `pixel_values` and every label's token ids
are computed once, here, instead of redone on every single `__getitem__` call across every
epoch. With the dataset now bigger (Phase 1's expansions) and training running for many
more epochs (Phase 2), repeating PIL-decode + resize + tokenize per access would be a real,
avoidable cost — see the profiling section in Phase 3 for what that would have looked like.

In [16]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader


def resolve_image_path(data_dir, rel_path):
    """labels.csv mixes two path conventions (Week 1 vs. Week 3 -- see the intro cell).
    Try the direct join first (covers Week 3's ~110 'synthetic' rows, stored relative to
    DATA_DIR), then fall back to stripping a redundant leading 'data/' (covers Week 1's
    ~153 'books'/'newspaper'/'other' rows, stored relative to DATA_DIR's parent). Recovers
    all 263 rows instead of 110. Reused in Step 3 Extra's prediction grid, which hits the
    exact same path-joining logic when it maps results back to image files.
    """
    direct = os.path.join(data_dir, rel_path)
    if os.path.isfile(direct):
        return direct
    if rel_path.startswith("data/"):
        stripped = os.path.join(data_dir, rel_path[len("data/"):])
        if os.path.isfile(stripped):
            return stripped
    return direct  # neither resolves -- caller's exists-check reports this one as genuinely missing


class UrduOCRDataset(Dataset):
    """PyTorch Dataset over labels.csv, skipping rows with no image on disk.

    Each item returns TrOCR-ready pixel_values and tokenized labels, with padding
    positions set to -100 so the loss function ignores them (the Week-3-vs-official-guide
    fix noted above). pixel_values/labels are computed once here in __init__ rather than
    on every __getitem__ call -- see the note in the Step 2 intro above for why.
    """

    def __init__(self, csv_path, processor, data_dir, max_length=MAX_LENGTH):
        data = pd.read_csv(csv_path)
        resolved = data["image"].apply(lambda p: resolve_image_path(data_dir, p))
        has_file = resolved.apply(os.path.isfile)
        n_missing = int((~has_file).sum())
        if n_missing:
            print(f"Skipping {n_missing} rows with no image on disk (checked both path conventions)")
        self.data = data[has_file].reset_index(drop=True)
        self.max_length = max_length
        resolved_paths = resolved[has_file].reset_index(drop=True)
        print(f"Dataset loaded: {len(self.data)} samples")

        pad_id = processor.tokenizer.pad_token_id
        self._pixel_values, self._labels = [], []
        for i in range(len(self.data)):
            image = Image.open(resolved_paths.iloc[i]).convert("RGB")
            pv = processor(image, return_tensors="pt").pixel_values.squeeze()
            # Padding/truncating by hand (slice + pad with pad_token_id) instead of passing
            # padding="max_length", truncation=True straight to the tokenizer: verified in a
            # sandbox that some tokenizers/transformers version pairs raise
            # `TypeError: enable_truncation() got an unexpected keyword argument 'direction'`
            # on that call path. Slicing a plain Python list never depends on that internal API.
            raw_ids = processor.tokenizer(str(self.data.iloc[i]["text"])).input_ids[: self.max_length]
            ids = raw_ids + [pad_id] * (self.max_length - len(raw_ids))
            ids = [t if t != pad_id else -100 for t in ids]
            self._pixel_values.append(pv)
            self._labels.append(torch.tensor(ids))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return {"pixel_values": self._pixel_values[idx], "labels": self._labels[idx]}


dataset = UrduOCRDataset(LABELS_PATH, processor, data_dir=DATA_DIR)  # now includes Phase 1's expansions

# Stratified split by category (roughly 70/15/15 train/val/test) instead of the handout's
# plain 80/20 random split. Two changes from the original:
#   1. A plain random split doesn't guarantee train and test see the same mix of
#      books/newspaper/other/synthetic -- with the path-bug fix now bringing the three
#      real-world categories back into play alongside synthetic, that mix matters more
#      than it did when only synthetic images were reachable.
#   2. There's now a validation slice, separate from the test set, used only to monitor
#      training and pick the best checkpoint (Step 2 Extra: Checkpoint to Disk, below).
#      The test set stays untouched until Step 3.
# Fixed seed for the same reason as before: an identical split across sessions is required
# for checkpoint-resume to never leak val/test images into training.
torch.manual_seed(42)
categories = dataset.data["category"].fillna("uncategorized")
train_indices, val_indices, test_indices = [], [], []
for cat in categories.unique():
    idx = categories.index[categories == cat].to_numpy()
    idx = idx[torch.randperm(len(idx)).numpy()]
    n_test = max(1, int(round(0.15 * len(idx))))
    n_val = max(1, int(round(0.15 * len(idx))))
    test_indices.extend(idx[:n_test].tolist())
    val_indices.extend(idx[n_test:n_test + n_val].tolist())
    train_indices.extend(idx[n_test + n_val:].tolist())

train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset = torch.utils.data.Subset(dataset, val_indices)
test_dataset = torch.utils.data.Subset(dataset, test_indices)
print(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}")
print("Train category mix:", categories.loc[train_indices].value_counts().to_dict())
print("Test category mix:", categories.loc[test_indices].value_counts().to_dict())
if len(val_dataset) < 30:
    print(f"\nNote: the validation slice has only {len(val_dataset)} samples -- CER "
          "measured on this few examples carries real epoch-to-epoch variance. Read the "
          "trend across several EVAL_EVERY checkpoints (see the diagnostic dashboard after "
          "the loss chart in Phase 2), not any single epoch's number.")
else:
    print(f"\nValidation slice: {len(val_dataset)} samples -- large enough that "
          "single-checkpoint CER is reasonably trustworthy on its own.")

Dataset loaded: 3280 samples
Train: 2296  Val: 492  Test: 492
Train category mix: {'public_utrset_real': 2100, 'synthetic': 78, 'synthetic_v2': 62, 'uncategorized': 56}
Test category mix: {'public_utrset_real': 450, 'synthetic': 16, 'synthetic_v2': 14, 'uncategorized': 12}

Validation slice: 492 samples -- large enough that single-checkpoint CER is reasonably trustworthy on its own.


In [17]:
import pandas as pd, os
from PIL import Image

df = pd.read_csv(LABELS_PATH)

def get_aspect(row):
    path = resolve_image_path(DATA_DIR, row["image"])
    if not os.path.isfile(path):
        return None
    w, h = Image.open(path).size
    return w / h

df["aspect"] = df.apply(get_aspect, axis=1)
# single lines are always wide (aspect > ~3); full pages are portrait/near-square
suspects = df[df["aspect"].notna() & (df["aspect"] < 2.0)]
print(f"{len(suspects)} / {len(df)} rows look like full pages, not single lines:")
print(suspects[["image", "text", "aspect"]].to_string())

232 / 3280 rows look like full pages, not single lines:
                                                     image                text    aspect
60                           data/raw/synthetic/urdu_1.png    پاکستان زندہ باد  1.924370
285   raw/public_utrset_real/public_utrset_real_000005.jpg                 (۲۰  1.000000
303   raw/public_utrset_real/public_utrset_real_000023.jpg                کنیر  1.571429
308   raw/public_utrset_real/public_utrset_real_000028.jpg                (۳۱)  1.150000
320   raw/public_utrset_real/public_utrset_real_000040.jpg                 ۵۴۰  1.600000
321   raw/public_utrset_real/public_utrset_real_000041.jpg            باقاعدگی  1.357143
322   raw/public_utrset_real/public_utrset_real_000042.jpg            ۲۲۱   ''  1.924812
332   raw/public_utrset_real/public_utrset_real_000052.jpg                (۲۲)  1.171875
349   raw/public_utrset_real/public_utrset_real_000069.jpg                 ۵۲۶  1.368421
365   raw/public_utrset_real/public_utrset_real_000085

In [18]:
import pandas as pd, shutil, datetime

df = pd.read_csv(LABELS_PATH)
bad_prefixes = ("data/raw/books/", "data/raw/newspaper/")
before = len(df)
df_clean = df[~df["image"].str.startswith(bad_prefixes)]

backup_path = LABELS_PATH + f".backup-{datetime.datetime.now():%Y%m%d-%H%M%S}"
shutil.copy(LABELS_PATH, backup_path)
df_clean.to_csv(LABELS_PATH, index=False)

print(f"Removed {before - len(df_clean)} rows -- {before} -> {len(df_clean)}")
print(df_clean["category"].fillna("uncategorized").value_counts())

Removed 0 rows -- 3280 -> 3280
category
public_utrset_real    3000
synthetic              110
synthetic_v2            90
uncategorized           80
Name: count, dtype: int64


In [19]:
# Validation report -- makes explicit what the Dataset class above did implicitly.
n_total_labels = len(pd.read_csv(LABELS_PATH))
n_loaded = len(dataset)
n_resolved_direct = sum(1 for p in dataset.data["image"] if os.path.isfile(os.path.join(DATA_DIR, p)))
n_resolved_fallback = n_loaded - n_resolved_direct
cache_bytes = sum(pv.element_size() * pv.nelement() for pv in dataset._pixel_values) + \
              sum(lb.element_size() * lb.nelement() for lb in dataset._labels)

print("=== Phase 1 validation report ===")
print(f"labels.csv rows:              {n_total_labels}")
print(f"Resolved and loaded:          {n_loaded}  ({n_loaded / n_total_labels:.0%})")
print(f"  - resolved directly:        {n_resolved_direct}")
print(f"  - resolved via fallback:    {n_resolved_fallback}  (older path convention -- see resolve_image_path)")
print(f"Dropped (unresolvable):       {n_total_labels - n_loaded}")
print(f"Cached tensor memory:         {cache_bytes / 1e6:.1f} MB")
print(f"Category mix:                 {dataset.data['category'].fillna('uncategorized').value_counts().to_dict()}")
assert n_loaded == n_resolved_direct + n_resolved_fallback
if n_total_labels - n_loaded > 0:
    print(f"\n{n_total_labels - n_loaded} row(s) genuinely have no matching image on disk -- "
          "worth a manual check if that number looks larger than expected.")

=== Phase 1 validation report ===
labels.csv rows:              3280
Resolved and loaded:          3280  (100%)
  - resolved directly:        3200
  - resolved via fallback:    80  (older path convention -- see resolve_image_path)
Dropped (unresolvable):       0
Cached tensor memory:         5809.0 MB
Category mix:                 {'public_utrset_real': 3000, 'synthetic': 110, 'synthetic_v2': 90, 'uncategorized': 80}


---
# Phase 2: Pretrained Model Fine-Tuning

## 2.1 Initialize the Pretrained Checkpoint

TrOCR combines a vision encoder (reads the image) with a text decoder (outputs
characters). `MODEL_NAME` was set in Phase 1 to `microsoft/trocr-base-printed`, the
checkpoint the handout specifies.

**Worth knowing before fine-tuning starts:** this model's text decoder is RoBERTa,
pretrained on English text — it has never seen Urdu script. The vision encoder's job
(reading strokes/shapes from pixels) transfers across scripts reasonably well, but the
decoder has to learn Urdu's token vocabulary essentially from scratch during fine-tuning.
That's a big part of why this notebook needs far more than a handful of epochs to produce
non-garbage output, and why accuracy has a real ceiling worth watching honestly rather
than assuming away.

If "domain-specific" is meant loosely (i.e. a checkpoint that's actually been exposed to
Urdu before), there's a community-fine-tuned option on the Hub —
`cxfajar197/urdu-ocr` — usable as a drop-in `MODEL_NAME` swap since it's the same
`VisionEncoderDecoderModel` architecture. Not vetted here for quality/license/provenance,
just flagged as worth investigating if you have flexibility on the base checkpoint;
sticking with the handout's `trocr-base-printed` below to keep this notebook's
checkpoints/resume state compatible with the last two audits.

In [20]:
from transformers import VisionEncoderDecoderModel

try:
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
except Exception as e:
    raise RuntimeError(
        f"Couldn't load '{MODEL_NAME}' from Hugging Face Hub: {e}. "
        "Check your internet connection and that huggingface.co is reachable."
    ) from e

model = model.to(device)

# Configure model for generation (standard TrOCR fine-tuning setup)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id  # NEW -- see the fix notes below cell 2.1
model.config.vocab_size = model.config.decoder.vocab_size

print("Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


## 2.2 Checkpoint, Resume, and Train

*(Not in the handout.)* A Codespace can still disconnect mid-training -- an idle timeout, a network hiccup, an accidentally closed browser tab -- which normally means starting the full run over from epoch 0. The cell below checks the `checkpoints/` folder next to your data for a checkpoint from a previous run and, if one exists, resumes from the next epoch instead of the beginning. The training loop after it saves a checkpoint (model weights, optimizer state, and loss history) to disk at the end of every epoch, overwriting the previous one so it doesn't pile up. This only stays correct because of the fixed seed added above -- it keeps the train/test split identical across sessions, so resuming never leaks old training images into the test set.

**Two additions from the audit pass:** (1) the saved checkpoint now also records the
dataset/train size it was written against, and refuses to silently resume if those don't
match what's loaded right now -- this project's very first `checkpoints/` folder was
written by the *pre-fix* pipeline (110 samples, not 263), and blindly resuming from it is
exactly what made the original run report "already trained" without training at all. If you
see the mismatch warning below, delete the old `checkpoints/` folder. (2) every
`EVAL_EVERY` epochs the loop now measures CER on the validation slice (not the test set)
and keeps a separate `best_model/` checkpoint -- with `NUM_EPOCHS` raised well past 3,
there's no guarantee the *last* epoch is the *best* one, and the original notebook had no
way to tell.

In [21]:
# NOTE: the handout says `from transformers import AdamW` -- that raises ImportError on
# transformers==4.57.6 (verified in a sandbox). Transformers dropped its own AdamW in favor
# of PyTorch's; torch.optim.AdamW is the standard drop-in replacement.
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import json

# --- Hyperparameters (revised during the audit pass -- see the intro cell for the numbers
#     behind these choices) ---
BATCH_SIZE = 8        # was 4. TrOCR-base at 384x384 comfortably fits a bigger batch on a
                       # T4's 16GB, especially with mixed precision below; fewer, bigger
                       # batches also means less DataLoader/Python-loop overhead per epoch.
                       # Lower this back to 4 if the OOM handler below tells you to.
LEARNING_RATE = 5e-5   # unchanged -- a standard fine-tuning LR for this model size. The
                       # original problem wasn't the LR value, it was too few total steps
                       # (see NUM_EPOCHS) and no warmup (see the scheduler below).
NUM_EPOCHS = 40        # was 20 (before that, 3). The 20-epoch run's loss chart shows no
                       # plateau -- still dropping at epoch 20, no sign of convergence -- and
                       # CER improved from 2.06 to 1.365 over the second half of that run alone.
                       # Doubling to 40 gives real room to keep closing that gap. If a session
                       # disconnects partway, the resume logic below picks up exactly where it
                       # left off -- no need to restart.
EVAL_EVERY = 2         # compute validation CER (and possibly update best_model/) every N
                       # epochs. Generation is much slower than a forward pass, so doing
                       # this every single epoch would meaningfully slow the loop down.
USE_AMP = device == "cuda"  # mixed precision: real speedup + lower memory on a T4, no-op on CPU.

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=(device == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                         num_workers=2, pin_memory=(device == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=(device == "cuda"))

CHECKPOINT_DIR = os.path.join(DATA_DIR, "checkpoints")
BEST_MODEL_DIR = os.path.join(DATA_DIR, "best_model")
CHECKPOINT_STATE_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint_state.json")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

total_steps = len(train_loader) * NUM_EPOCHS


def fresh_optimizer_and_scheduler(m):
    opt = AdamW(m.parameters(), lr=LEARNING_RATE)
    sched = get_linear_schedule_with_warmup(
        opt, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
    )
    return opt, sched


resume_ok = False
if os.path.isfile(CHECKPOINT_STATE_PATH):
    with open(CHECKPOINT_STATE_PATH) as f:
        ckpt_state = json.load(f)
    # Guard against silently resuming from a checkpoint written against a different dataset
    # -- see the note in the markdown cell above.
    if ckpt_state.get("dataset_size") != len(dataset) or ckpt_state.get("train_size") != len(train_dataset):
        print(f"Found a checkpoint at {CHECKPOINT_DIR}, but it doesn't match the current "
              f"dataset (saved dataset_size={ckpt_state.get('dataset_size')}, "
              f"train_size={ckpt_state.get('train_size')} vs. current {len(dataset)}, "
              f"{len(train_dataset)}). Treating it as stale and starting fresh -- delete "
              "the checkpoints/ folder to silence this check.")
    else:
        resume_ok = True

if resume_ok:
    start_epoch = ckpt_state["last_completed_epoch"] + 1
    loss_history = ckpt_state["loss_history"]
    epoch_avg_losses = ckpt_state["epoch_avg_losses"]
    best_val_cer = ckpt_state.get("best_val_cer", float("inf"))
    val_history = ckpt_state.get("val_history", [])  # NEW -- see Cell 2.3b
    print(f"Found a valid checkpoint through epoch {start_epoch} at {CHECKPOINT_DIR} -- reloading model weights.")
    model = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT_DIR).to(device)
    optimizer, scheduler = fresh_optimizer_and_scheduler(model)
    optimizer.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "optimizer.pt"), map_location=device))
    if os.path.isfile(os.path.join(CHECKPOINT_DIR, "scheduler.pt")):
        scheduler.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "scheduler.pt"), map_location=device))
else:
    print("No valid checkpoint found -- starting training from scratch.")
    optimizer, scheduler = fresh_optimizer_and_scheduler(model)
    start_epoch = 0
    loss_history = []
    epoch_avg_losses = []
    best_val_cer = float("inf")
    val_history = []  # NEW -- per-checkpoint (loss, CER, breakdown) log, see Cell 2.3b

print(f"Training batches per epoch: {len(train_loader)}  (total planned steps: {total_steps})")
if start_epoch >= NUM_EPOCHS:
    print(f"All {NUM_EPOCHS} epochs already completed per the saved checkpoint -- nothing left to train.")
else:
    print(f"Ready to train epochs {start_epoch + 1} through {NUM_EPOCHS}.")

Found a valid checkpoint through epoch 11 at /content/drive/MyDrive/Urdu-OCR-Project/urdu-ocr-repo/SI26-Week1/data/checkpoints -- reloading model weights.
Training batches per epoch: 287  (total planned steps: 11480)
Ready to train epochs 12 through 40.


In [ ]:
# This cell will take noticeably longer than the original 20-40 minute estimate -- many
# more epochs, plus a validation pass every EVAL_EVERY epochs. Safe to re-run after a
# disconnect: it continues from the last saved checkpoint instead of retraining from epoch
# 0 (see the resume/staleness check above).

import jiwer  # also used again in evaluation below
import time  # profiling -- see Phase 3

scaler = torch.amp.GradScaler(device="cuda", enabled=USE_AMP)

# Phase 3 profiling: how much of each epoch is data loading vs. actual GPU compute.
# Cheap to collect (a couple of perf_counter() calls per step) and answers "where's
# the bottleneck" with a real number instead of a guess.
data_time_total = 0.0
compute_time_total = 0.0


def run_generation_eval(loader, m):
    """Beam-search generation + a full character-level error breakdown over a loader.
    Used for the periodic validation check here, and reused as-is for the final Step 3
    evaluation.

    Returns a dict instead of a bare CER float -- CER alone can't say *why* it's high.
    Insertions/deletions/substitutions plus the hypothesis-vs-reference length ratio can:
    a CER>1 driven mostly by insertions with hyp much longer than ref points at a
    generation-side runaway (repetition/hallucination, or the decoder not yet reliably
    predicting EOS during free-running generation) rather than the model being
    confidently wrong character-for-character."""
    m.eval()
    refs, hyps = [], []
    with torch.no_grad():
        for batch in loader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].clone()
            labels[labels == -100] = processor.tokenizer.pad_token_id
            generated_ids = m.generate(pixel_values, max_length=MAX_LENGTH, num_beams=4,
                                repetition_penalty=1.3, no_repeat_ngram_size=3)
            hyps.extend(processor.batch_decode(generated_ids, skip_special_tokens=True))
            refs.extend(processor.batch_decode(labels, skip_special_tokens=True))
    m.train()
    if not refs:
        return {"cer": float("nan")}
    out = jiwer.process_characters(refs, hyps)
    ref_chars = sum(len(r) for r in refs)
    hyp_chars = sum(len(h) for h in hyps)
    n_edits = out.insertions + out.deletions + out.substitutions
    return {
        "cer": out.cer,
        "insertions": out.insertions,
        "deletions": out.deletions,
        "substitutions": out.substitutions,
        "hits": out.hits,
        "ref_chars": ref_chars,
        "hyp_chars": hyp_chars,
        "length_ratio": (hyp_chars / ref_chars) if ref_chars else float("nan"),
        "insertion_share": (out.insertions / n_edits) if n_edits else float("nan"),
    }


for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    total_loss = 0
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 30)
    epoch_data_time = 0.0
    epoch_compute_time = 0.0
    _t_prev = time.perf_counter()
    for batch_idx, batch in enumerate(train_loader):
        _t_fetched = time.perf_counter()
        epoch_data_time += _t_fetched - _t_prev

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        try:
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                outputs = model(pixel_values=pixel_values, labels=labels)
                loss = outputs.loss

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # Gradient clipping: standard stabilizer for full-parameter fine-tuning of a
            # large pretrained model on data far outside its original training distribution.
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        except torch.cuda.OutOfMemoryError:
            print(f"  Batch {batch_idx}: ran out of GPU memory. Lower BATCH_SIZE in the "
                  "cell above and re-run from there.")
            raise

        _t_prev = time.perf_counter()
        epoch_compute_time += _t_prev - _t_fetched

        total_loss += loss.item()
        loss_history.append(loss.item())
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f} | "
                  f"LR: {scheduler.get_last_lr()[0]:.2e}")

    avg_loss = total_loss / len(train_loader)
    epoch_avg_losses.append(avg_loss)
    print(f"Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}")
    data_time_total += epoch_data_time
    compute_time_total += epoch_compute_time
    _epoch_wall = epoch_data_time + epoch_compute_time
    if _epoch_wall > 0:
        print(f"  Profiling: {epoch_data_time:.1f}s data loading, {epoch_compute_time:.1f}s compute ("
              f"{100 * epoch_data_time / _epoch_wall:.0f}% data-bound) | {len(train_dataset) / _epoch_wall:.1f} samples/sec")

    # Checkpoint to disk (Not in the handout) -- overwrites the previous epoch's save.
    model.save_pretrained(CHECKPOINT_DIR)
    processor.save_pretrained(CHECKPOINT_DIR)
    torch.save(optimizer.state_dict(), os.path.join(CHECKPOINT_DIR, "optimizer.pt"))
    torch.save(scheduler.state_dict(), os.path.join(CHECKPOINT_DIR, "scheduler.pt"))

    # Extra: validate every EVAL_EVERY epochs and keep a separate best_model/ checkpoint --
    # NUM_EPOCHS is now high enough that the last epoch isn't guaranteed to be the best one.
    if (epoch + 1) % EVAL_EVERY == 0 or (epoch + 1) == NUM_EPOCHS:
        val_metrics = run_generation_eval(val_loader, model)
        val_cer = val_metrics["cer"]
        print(f"Validation CER after epoch {epoch + 1}: {val_cer:.3f} (best so far: {best_val_cer:.3f})")
        print(f"  breakdown -- insertions: {val_metrics['insertions']}, "
              f"deletions: {val_metrics['deletions']}, substitutions: {val_metrics['substitutions']} | "
              f"hyp/ref length ratio: {val_metrics['length_ratio']:.2f} | "
              f"insertion share of edits: {val_metrics['insertion_share']:.0%}")

        val_history.append({"epoch": epoch + 1, "avg_loss": avg_loss, **val_metrics})

        # Diagnostic: pairs THIS checkpoint's loss/CER movement against the previous one.
        # Overfitting requires loss to still be low/flat while val error rises -- loss
        # still falling fast is the opposite signature, and points at the decoder still
        # being early in learning (exposure bias / not-yet-reliable EOS prediction during
        # free-running generation) rather than memorization. See Cell 2.3b for the full
        # history plotted together.
        if len(val_history) >= 2:
            prev, cur = val_history[-2], val_history[-1]
            loss_falling = cur["avg_loss"] < prev["avg_loss"]
            cer_rising = cur["cer"] > prev["cer"]
            if loss_falling and cer_rising:
                print("  Diagnosis: loss still falling while CER rose -- a genuine "
                      "overfitting signal at this checkpoint.")
            elif loss_falling and not cer_rising:
                print("  Diagnosis: loss and CER both improving -- healthy, keep training.")
            else:
                print("  Diagnosis: loss isn't falling right now either -- this CER move "
                      "is more likely generation instability than overfitting (overfitting "
                      "needs the loss to still be dropping).")

        if val_cer < best_val_cer:
            best_val_cer = val_cer
            model.save_pretrained(BEST_MODEL_DIR)
            processor.save_pretrained(BEST_MODEL_DIR)
            print(f"  New best -- saved to {BEST_MODEL_DIR}")

    with open(CHECKPOINT_STATE_PATH, "w") as f:
        json.dump({
            "last_completed_epoch": epoch,
            "loss_history": loss_history,
            "epoch_avg_losses": epoch_avg_losses,
            "best_val_cer": best_val_cer,
            "val_history": val_history,  # NEW -- see Cell 2.3b
            "dataset_size": len(dataset),
            "train_size": len(train_dataset),
        }, f)
    print(f"Checkpoint saved to disk ({CHECKPOINT_DIR})")

print("\nTraining complete!")
print(f"Training loss went from {loss_history[0]:.4f} (first batch) to {loss_history[-1]:.4f} (last batch)")
if best_val_cer < float("inf"):
    print(f"Best validation CER seen during training: {best_val_cer:.3f} (checkpoint saved to {BEST_MODEL_DIR})")

# Carried into Phase 3's profiling report.
print(f"\nTotal training time -- data loading: {data_time_total:.1f}s, compute: {compute_time_total:.1f}s")


Epoch 12/40
------------------------------


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Batch 0/287 | Loss: 0.7138 | LR: 3.89e-05
  Batch 10/287 | Loss: 0.5459 | LR: 3.88e-05
  Batch 20/287 | Loss: 0.9511 | LR: 3.88e-05
  Batch 30/287 | Loss: 0.5242 | LR: 3.87e-05
  Batch 40/287 | Loss: 0.3861 | LR: 3.87e-05
  Batch 50/287 | Loss: 0.5184 | LR: 3.86e-05
  Batch 60/287 | Loss: 0.6721 | LR: 3.86e-05
  Batch 70/287 | Loss: 0.5519 | LR: 3.85e-05
  Batch 80/287 | Loss: 0.4901 | LR: 3.85e-05
  Batch 90/287 | Loss: 0.4439 | LR: 3.84e-05
  Batch 100/287 | Loss: 0.5027 | LR: 3.84e-05
  Batch 110/287 | Loss: 0.6775 | LR: 3.84e-05
  Batch 120/287 | Loss: 1.2069 | LR: 3.83e-05
  Batch 130/287 | Loss: 0.4321 | LR: 3.83e-05
  Batch 140/287 | Loss: 1.0497 | LR: 3.82e-05
  Batch 150/287 | Loss: 0.5097 | LR: 3.82e-05
  Batch 160/287 | Loss: 0.6576 | LR: 3.81e-05
  Batch 170/287 | Loss: 0.3978 | LR: 3.81e-05
  Batch 180/287 | Loss: 0.4055 | LR: 3.80e-05
  Batch 190/287 | Loss: 0.6144 | LR: 3.80e-05
  Batch 200/287 | Loss: 0.4648 | LR: 3.79e-05
  Batch 210/287 | Loss: 0.3321 | LR: 3.79e-05

## 2.3 Visualize Training Progress

*(Not in the handout.)* The submission asks for a screenshot of the training output showing loss decreasing — an actual chart makes that point far more clearly than a screenshot of scrolling console text, and it's this cell's output that best supports the required `'Training loss went from X to X'` line.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(loss_history, color="#4C72B0", linewidth=1.2)
batches_per_epoch = len(train_loader)
for e in range(1, NUM_EPOCHS):
    axes[0].axvline(e * batches_per_epoch, color="#999999", linestyle=":", linewidth=1)
axes[0].set_title("Training Loss per Batch")
axes[0].set_xlabel("Batch (cumulative across epochs)")
axes[0].set_ylabel("Loss")

axes[1].plot(range(1, NUM_EPOCHS + 1), epoch_avg_losses, marker="o", color="#C44E52")
axes[1].set_title("Average Loss per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Average Loss")
axes[1].set_xticks(range(1, NUM_EPOCHS + 1))
for i, v in enumerate(epoch_avg_losses):
    axes[1].annotate(f"{v:.3f}", (i + 1, v), textcoords="offset points", xytext=(0, 8), ha="center")

plt.tight_layout()
plt.savefig("training_loss.png", dpi=150)
plt.show()
print("Saved to training_loss.png -- attach this for the 'screenshot of training output' requirement.")

## 2.3b Loss vs. CER Diagnostic Dashboard

*(Not in the handout -- added to answer "is this overfitting?" with data instead of a
guess.)* CER alone can't distinguish real overfitting from an undertrained decoder still
finding its footing, or from noise on a small validation slice. This pairs training loss
with validation CER -- and its insertion/deletion/substitution breakdown -- at every
`EVAL_EVERY` checkpoint recorded in `val_history` (Cell 2.2), and prints the same
per-checkpoint diagnosis the training loop already printed live, all in one place for
the writeup.

In [ ]:
import matplotlib.pyplot as plt

if not val_history:
    print("No validation checkpoints recorded yet -- val_history is empty. This fills in "
          "once training runs at least EVAL_EVERY epochs (Cell 2.2).")
else:
    epochs_ = [v["epoch"] for v in val_history]
    losses_ = [v["avg_loss"] for v in val_history]
    cers_ = [v["cer"] for v in val_history]

    fig, ax1 = plt.subplots(figsize=(9, 4.5))
    ax1.plot(epochs_, losses_, color="#4C72B0", marker="o", label="Train loss (epoch avg)")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Training loss", color="#4C72B0")
    ax1.tick_params(axis="y", labelcolor="#4C72B0")

    ax2 = ax1.twinx()
    ax2.plot(epochs_, cers_, color="#C44E52", marker="s", label="Validation CER")
    ax2.axhline(1.0, color="#C44E52", linestyle=":", linewidth=1, alpha=0.6)
    ax2.set_ylabel("Validation CER", color="#C44E52")
    ax2.tick_params(axis="y", labelcolor="#C44E52")

    fig.suptitle("Training loss vs. validation CER, by checkpoint")
    fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.88))
    fig.tight_layout()
    plt.savefig("loss_vs_cer.png", dpi=150)
    plt.show()
    print("Saved to loss_vs_cer.png.\n")

    print("=== Diagnostic table ===")
    header = f"{'Epoch':>6} {'TrainLoss':>10} {'ValCER':>8} {'InsShare':>9} {'Hyp/RefLen':>11}  Diagnosis"
    print(header)
    for i, v in enumerate(val_history):
        if i == 0:
            diag = "(first checkpoint -- no prior point to compare against)"
        else:
            prev = val_history[i - 1]
            loss_falling = v["avg_loss"] < prev["avg_loss"]
            cer_rising = v["cer"] > prev["cer"]
            if loss_falling and cer_rising:
                diag = "overfitting signal"
            elif loss_falling and not cer_rising:
                diag = "healthy -- keep training"
            else:
                diag = "loss flat/rising -- CER move is noise/instability, not overfitting"
        row = (f"{v['epoch']:>6} {v['avg_loss']:>10.4f} {v['cer']:>8.3f} "
               f"{v['insertion_share']:>9.0%} {v['length_ratio']:>11.2f}  {diag}")
        print(row)

    n_overfit_signals = sum(
        1 for i in range(1, len(val_history))
        if val_history[i]["avg_loss"] < val_history[i - 1]["avg_loss"]
        and val_history[i]["cer"] > val_history[i - 1]["cer"]
    )
    print(f"\n{n_overfit_signals} / {len(val_history) - 1} checkpoint-to-checkpoint "
          "transitions look like genuine overfitting (loss still falling, CER rising).")
    print(f"Validation slice size: {len(val_dataset)} samples -- "
          + ("small enough that single-epoch CER swings carry real noise."
             if len(val_dataset) < 30 else
             "large enough that these numbers are reasonably trustworthy on their own."))
    if any(v["insertion_share"] > 0.5 for v in val_history if v["insertion_share"] == v["insertion_share"]):
        print("\nAt least one checkpoint has insertion-dominated errors (>50% of edits) -- "
              "consistent with the exposure-bias / repetition pattern already diagnosed "
              "in the intro cell (free-running generation compounding errors that "
              "teacher-forced training loss never penalizes).")

## 2.4 Evaluate Your Model

After training, this tests the model on images it has never seen — the test split from above. `model.eval()` turns off weight updates during this step.

Beyond the handout's exact-match accuracy, this also reports **Character Error Rate (CER)** and **Word Error Rate (WER)** — exact-match is strict (one wrong character anywhere in the line counts as fully wrong), while CER/WER are the standard OCR metrics and give a more graded sense of how close predictions are.

In [ ]:
import jiwer

# Use the best validation-CER checkpoint from training (Step 2 Extra: Checkpoint to Disk)
# if one was saved -- the model in memory right now is whatever epoch training happened to
# stop on, which isn't guaranteed to be the best one now that NUM_EPOCHS is more than a couple.
if os.path.isdir(BEST_MODEL_DIR) and os.listdir(BEST_MODEL_DIR):
    print(f"Loading best checkpoint by validation CER from {BEST_MODEL_DIR} for final evaluation.\n")
    model = VisionEncoderDecoderModel.from_pretrained(BEST_MODEL_DIR).to(device)
else:
    print("No best_model/ checkpoint found -- evaluating whatever model is currently in memory.\n")

model.eval()
print("=== Model Evaluation on Test Images ===\n")

results = []
with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id  # undo the training-time mask for decoding

        # num_beams=4: beam search instead of the handout's implicit greedy decoding
        # (num_beams=1). Greedy commits to the single best token at each step with no way
        # back; beam search keeps several candidate sequences alive and tends to produce
        # closer transcriptions for seq2seq generation tasks like this one.
        #
        # repetition_penalty/no_repeat_ngram_size: NEW. The previous run's predictions were
        # dominated by runaway repeated n-grams (e.g. "و�و�و�و�و") rather than clean-but-wrong
        # text -- a well-known seq2seq failure mode (exposure bias: teacher-forced training
        # loss doesn't directly penalize this, since it only ever sees true previous tokens,
        # not the model's own compounding mistakes during free-running generation). These two
        # params make the decoder actively avoid repeating itself; they don't fix the
        # underlying undertraining, but they stop repetition loops from dominating the output
        # and drowning out whatever the model has actually learned.
        generated_ids = model.generate(pixel_values, max_length=MAX_LENGTH, num_beams=4,
                                        repetition_penalty=1.3, no_repeat_ngram_size=3)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
        actual_text = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            pred, actual = pred.strip(), actual.strip()
            results.append({
                "predicted": pred,
                "actual": actual,
                "exact_match": pred == actual,
                "cer": jiwer.cer(actual, pred) if actual else None,
            })
            print(f"Predicted: {pred}")
            print(f"Actual: {actual}")
            print()

correct = sum(r["exact_match"] for r in results)
total = len(results)
accuracy = (correct / total) * 100 if total > 0 else 0

refs = [r["actual"] for r in results]
hyps = [r["predicted"] for r in results]
overall_cer = jiwer.cer(refs, hyps) if refs else float("nan")
overall_wer = jiwer.wer(refs, hyps) if refs else float("nan")

print(f"Accuracy: {accuracy:.1f}% ({correct}/{total} correct)")
print(f"Character Error Rate (CER): {overall_cer:.3f}")
print(f"Word Error Rate (WER): {overall_wer:.3f}")

if refs:
    _breakdown = jiwer.process_characters(refs, hyps)
    _n_edits = _breakdown.insertions + _breakdown.deletions + _breakdown.substitutions
    _hyp_chars = sum(len(h) for h in hyps)
    _ref_chars = sum(len(r) for r in refs)
    print(f"CER breakdown -- insertions: {_breakdown.insertions}, "
          f"deletions: {_breakdown.deletions}, substitutions: {_breakdown.substitutions}"
          + (f" (insertion share: {_breakdown.insertions / _n_edits:.0%})" if _n_edits else ""))
    print(f"Hyp/ref length ratio: {(_hyp_chars / _ref_chars) if _ref_chars else float('nan'):.2f} "
          "(>1 means predictions run longer than ground truth on average)")

## 2.5 Automatically Find the Worst Predictions

*(Not in the handout.)* The handout asks you to note 3-5 examples where the model got it wrong for Week 5. This pulls the worst ones by CER automatically instead of scrolling through the printed output above by hand.

In [ ]:
print("=== Examples where the model got it wrong (for your Week 5 discussion points) ===\n")

wrong = [r for r in results if not r["exact_match"]]
wrong_sorted = sorted(wrong, key=lambda r: (r["cer"] if r["cer"] is not None else 0), reverse=True)

if not wrong_sorted:
    print("No mistakes on the test set (or the test set is very small -- check the count above).")
else:
    for i, r in enumerate(wrong_sorted[:5], 1):
        print(f"{i}. Predicted: {r['predicted']}")
        print(f"   Actual:    {r['actual']}")
        print(f"   CER: {r['cer']:.3f}" if r["cer"] is not None else "   CER: n/a")
        print()

### 2.7 Save Evaluation Metrics and Predictions to Google Drive

This section saves the comprehensive evaluation metrics, all individual test predictions, and the top worst-performing predictions to Google Drive. This ensures that the results are persistently stored for future analysis and reporting, even if the Colab runtime disconnects.

In [ ]:
import json
import pandas as pd
import os
import datetime

# Define a directory in Google Drive to save the results
RESULTS_DIR = os.path.join(DRIVE_ROOT, "evaluation_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# 1. Save Evaluation Summary Metrics
metrics_summary = {
    "accuracy_pct": accuracy,
    "overall_cer": overall_cer,
    "overall_wer": overall_wer,
    "num_test_samples": total
}
metrics_summary_path_json = os.path.join(RESULTS_DIR, f"metrics_summary_{timestamp}.json")
with open(metrics_summary_path_json, "w") as f:
    json.dump(metrics_summary, f, indent=2, ensure_ascii=False)
print(f"Evaluation summary metrics saved to: {metrics_summary_path_json}")

# 2. Save All Test Predictions
df_all_predictions = pd.DataFrame(results)
all_predictions_path_csv = os.path.join(RESULTS_DIR, f"all_test_predictions_{timestamp}.csv")
all_predictions_path_json = os.path.join(RESULTS_DIR, f"all_test_predictions_{timestamp}.json")
df_all_predictions.to_csv(all_predictions_path_csv, index=False)
df_all_predictions.to_json(all_predictions_path_json, orient="records", indent=2, force_ascii=False)
print(f"All test predictions saved to: {all_predictions_path_csv} and {all_predictions_path_json}")

# 3. Save Worst-Performing Predictions
# `wrong_sorted` is available from cell-037
# Take the top 10 worst predictions, or fewer if not enough mistakes were made
top_n_worst = 10
df_worst_predictions = pd.DataFrame(wrong_sorted[:top_n_worst])
worst_predictions_path_csv = os.path.join(RESULTS_DIR, f"worst_predictions_{timestamp}.csv")
worst_predictions_path_json = os.path.join(RESULTS_DIR, f"worst_predictions_{timestamp}.json")
df_worst_predictions.to_csv(worst_predictions_path_csv, index=False)
df_worst_predictions.to_json(worst_predictions_path_json, orient="records", indent=2, force_ascii=False)
print(f"Worst-performing predictions saved to: {worst_predictions_path_csv} and {worst_predictions_path_json}")

print(f"\nAll evaluation results and predictions saved to: {RESULTS_DIR}")

## 2.6 Visualize Sample Predictions

*(Not in the handout.)* A grid of actual test images next to what the model predicted vs. the ground truth — green border for an exact match, red for a mismatch.

In [ ]:
import textwrap
import random
from PIL import ImageDraw, ImageFont

# Map results back to their source image paths. test_loader has shuffle=False, so it
# iterates test_dataset in the same order as test_dataset.indices -- verified in testing.
test_image_paths = [test_dataset.dataset.data.iloc[i]["image"] for i in test_dataset.indices]
assert len(results) == len(test_image_paths), (
    f"Got {len(results)} predictions but {len(test_image_paths)} test images -- "
    "re-run the evaluation cell above before this one."
)
for r, img_path in zip(results, test_image_paths):
    r["image"] = img_path


import urllib.request

# The caption font (Urdu Nastaliq) isn't bundled with this notebook -- download it once
# so the fallback font_path below (used since no Week-3 FONTS dict is in memory here)
# actually points at a real file instead of a path that was never created.
FONTS_CACHE_DIR = os.path.join(REPO_DIR, "fonts")
os.makedirs(FONTS_CACHE_DIR, exist_ok=True)
NASTALIQ_FONT_PATH = os.path.join(FONTS_CACHE_DIR, "NotoNastaliqUrdu.ttf")
if not os.path.isfile(NASTALIQ_FONT_PATH):
    FONT_URL = (
        "https://raw.githubusercontent.com/google/fonts/main/"
        "ofl/notonastaliqurdu/NotoNastaliqUrdu%5Bwght%5D.ttf"
    )
    urllib.request.urlretrieve(FONT_URL, NASTALIQ_FONT_PATH)
    print(f"Downloaded Urdu Nastaliq font to {NASTALIQ_FONT_PATH}")


def show_prediction_grid(results, data_dir, n=9, cols=3, seed=0,
                          font_path=FONTS["nastaliq"] if "FONTS" in dir() else "fonts/NotoNastaliqUrdu.ttf",
                          thumb_size=230, caption_h=140, font_size=15):
    """Grid of sample test predictions: image + predicted vs. actual text.

    Captions are drawn with PIL using RTL shaping (direction='rtl', language='ur'),
    the same approach used for the Week 3 dataset-sample grid, since matplotlib
    cannot shape Arabic-script text on its own.
    """
    if not results:
        print("No results to show.")
        return
    random.seed(seed)
    sample = random.sample(results, k=min(n, len(results)))

    n_cols = min(cols, len(sample))
    n_rows = -(-len(sample) // n_cols)
    cell_w, cell_h = thumb_size, thumb_size + caption_h
    row_h = int(font_size * 2.1)

    grid_img = Image.new("RGB", (n_cols * cell_w, n_rows * cell_h), (255, 255, 255))
    font = ImageFont.truetype(font_path, font_size)
    label_font = ImageFont.load_default()

    for i, r in enumerate(sample):
        row_c, col_c = divmod(i, n_cols)
        thumb = Image.open(resolve_image_path(data_dir, r["image"])).convert("RGB")  # audit fix: same path-convention issue as the Dataset class
        thumb.thumbnail((thumb_size - 10, thumb_size - 10))
        cell = Image.new("RGB", (cell_w, cell_h), (255, 255, 255))
        cell.paste(thumb, ((cell_w - thumb.width) // 2, (thumb_size - thumb.height) // 2))

        draw = ImageDraw.Draw(cell)
        y = thumb_size + 8
        for label, text, color in [("Pred:", r["predicted"], (30, 60, 150)), ("True:", r["actual"], (20, 20, 20))]:
            draw.text((8, y), label, font=label_font, fill=(120, 120, 120))
            y += 14
            wrapped = textwrap.wrap(str(text), width=22)[:1]
            line = wrapped[0] if wrapped else str(text)
            if len(str(text)) > 22:
                line = line + "…"
            bbox = draw.textbbox((0, 0), line, font=font, direction="rtl", language="ur")
            tw = bbox[2] - bbox[0]
            draw.text((min(cell_w - 8, tw + 8), y), line, font=font, fill=color,
                       direction="rtl", language="ur", anchor="ra")
            y += row_h

        border_color = (85, 168, 104) if r["exact_match"] else (196, 78, 82)
        draw.rectangle([0, 0, cell_w - 1, cell_h - 1], outline=border_color, width=4)
        grid_img.paste(cell, (col_c * cell_w, row_c * cell_h))

    plt.figure(figsize=(n_cols * 3.2, n_rows * 4.1))
    plt.imshow(grid_img)
    plt.axis("off")
    n_correct = sum(r["exact_match"] for r in sample)
    plt.title(f"Sample Predictions ({n_correct}/{len(sample)} exact match) — green = correct, red = mismatch")
    plt.tight_layout()
    plt.show()

show_prediction_grid(results, DATA_DIR, n=9)

## 2.8 Optional: Compare Alternative Pretrained Checkpoints

*(Not in the handout -- added per the "if the current architecture is inadequate" ask.)*
`microsoft/trocr-base-printed`'s decoder is RoBERTa, pretrained only on English -- that's
been the running theory for why this needs so much training. Worth knowing it's not just
this notebook's theory: multiple people independently hit the identical wall trying to
fine-tune TrOCR on Urdu/Arabic/Persian and landed on the same fix direction. From a
`microsoft/trocr-base-handwritten` discussion thread: *"i am trying to train it on RTL
languages like persian... i think the best approach would be to just train the
tokenizer"* -- and a later reply: *"I am also working on RTL Languages currently like
Urdu and Arabic."* That's the same diagnosis this notebook has been running on, from
people with no connection to this project.

Two checkpoints worth trying, both drop-in compatible (same `TrOCRProcessor` +
`VisionEncoderDecoderModel` API, just a different `MODEL_NAME`):

- **`cxfajar197/urdu-ocr`** -- already fine-tuned on Urdu specifically. Lowest risk,
  easiest comparison.
- **`RayR1/trocr-base-arabic-handwritten`** -- decoder tokenizer swapped to
  `CAMeL-Lab/bert-base-arabic-camelbert-ca` (a real Arabic-script vocabulary), base model
  `microsoft/trocr-large-handwritten`. Not Urdu-specific and it's *handwritten*-domain
  rather than printed, so it's not a perfect match either -- but a script-appropriate
  decoder vocabulary is exactly the fix that thread converged on, and this is a working
  example of someone having already done that surgery for the same base script.

Neither has been benchmarked against this project's own data by anyone involved in
building this notebook -- the cell below runs a **zero-shot check (no fine-tuning)**
against a handful of your own validation images, so you get a real number for your
actual data before deciding whether either is worth a full fine-tuning run. A better
zero-shot CER than a from-scratch `trocr-base-printed` result would suggest fine-tuning
the alternative could reach a given CER target in fewer epochs -- it's a starting-point
comparison, not a replacement for full training either way.

In [ ]:
from transformers import TrOCRProcessor as _AltProcessor, VisionEncoderDecoderModel as _AltModel

CANDIDATE_CHECKPOINTS = [
    "cxfajar197/urdu-ocr",
    "RayR1/trocr-base-arabic-handwritten",
]
N_PROBE_SAMPLES = 8  # small on purpose -- this is a quick zero-shot sanity check, not a full eval


def zero_shot_probe(checkpoint_name, val_subset, n=N_PROBE_SAMPLES):
    print(f"\nLoading {checkpoint_name} (zero-shot, no fine-tuning)...")
    try:
        alt_processor = _AltProcessor.from_pretrained(checkpoint_name)
        alt_model = _AltModel.from_pretrained(checkpoint_name).to(device)
    except Exception as e:
        print(f"  Couldn't load {checkpoint_name}: {e}")
        return None
    alt_model.eval()

    # Re-decode this notebook's own cached pixel_values/labels through the CANDIDATE's
    # processor/tokenizer -- pixel_values were preprocessed with the ORIGINAL model's
    # image processor in Phase 1, which may use different normalization than the
    # candidate expects, so this reloads raw images directly for a fair comparison.
    refs, hyps = [], []
    indices = list(range(len(val_subset)))[:n]
    with torch.no_grad():
        for idx in indices:
            row = dataset.data.iloc[val_subset.indices[idx]]
            img_path = resolve_image_path(DATA_DIR, row["image"])
            image = Image.open(img_path).convert("RGB")
            pixel_values = alt_processor(image, return_tensors="pt").pixel_values.to(device)
            generated_ids = alt_model.generate(pixel_values, max_length=MAX_LENGTH, num_beams=4,
                                                repetition_penalty=1.3, no_repeat_ngram_size=3)
            hyps.append(alt_processor.batch_decode(generated_ids, skip_special_tokens=True)[0])
            refs.append(row["text"])

    cer = jiwer.cer(refs, hyps)
    print(f"  Zero-shot CER on {len(refs)} samples: {cer:.3f}")
    for r, h in list(zip(refs, hyps))[:3]:
        print(f"    actual: {r!r}\n    pred:   {h!r}")
    del alt_model
    if device == "cuda":
        torch.cuda.empty_cache()
    return cer


print(f"For reference, this notebook's own fine-tuned model's test CER: see Phase 2's evaluation cell above.")
probe_results = {}
for ckpt in CANDIDATE_CHECKPOINTS:
    result = zero_shot_probe(ckpt, val_dataset)
    if result is not None:
        probe_results[ckpt] = result

if probe_results:
    print("\n=== Zero-shot comparison summary ===")
    for ckpt, cer in sorted(probe_results.items(), key=lambda kv: kv[1]):
        print(f"  {cer:.3f}  {ckpt}")
    print("\nTo actually fine-tune the best of these on your data: set MODEL_NAME to it "
          "near the top of Phase 2 (2.1), delete or rename the checkpoints/ and best_model/ "
          "folders (different architecture/vocab -- not resumable from the current "
          "checkpoint), and re-run from 2.1 onward.")

---
# Phase 3: Code Audit & Optimization

Mixed precision, the LR schedule + gradient clipping, and best-checkpoint-by-validation-CER
are already live in Phase 2 — see the overview cell for why they're built into the training
loop rather than added after the fact. This phase does four things that couldn't honestly
happen any earlier:

1. **Recap** what the original audit found and why it drove every choice above.
2. **Report** the data-loading-vs-compute profile Phase 2's training loop just collected.
3. **Probe** the actual GPU for a real batch-size ceiling, empirically, instead of a guess.
4. **Sanity-check** the learning rate with a cheap few-step comparison across candidates.
5. **Confirm** best-checkpoint selection worked -- which epoch actually won, and by how much.

## 3.1 Audit Recap

*(Context for why Phase 1/2 look the way they do -- skip if you've read the last two
audits already.)*

The original Week 4 notebook scored **0% accuracy (0/22)**, with every test prediction
decoding to the Unicode replacement character (`�`). Two compounding causes:

- **A path-convention bug in `labels.csv`** silently dropped 153 of 263 rows (58%) --
  fixed by `resolve_image_path()`'s fallback logic in Phase 1, now handling whatever mix
  of conventions Phase 1's own expansions plus the original data produce.
- **A 334M-parameter model getting ~66 total gradient steps** (88 training examples, 3
  epochs, batch size 4) -- nowhere near enough for a decoder that's never seen Urdu
  script (RoBERTa, English-only pretraining) to learn a new byte-level vocabulary from
  scratch. Phase 2 fixes the mechanics (much larger step count, warmup+decay, gradient
  clipping, mixed precision) but the English-only decoder pretraining is a real ceiling,
  not just a tuning problem -- worth watching the actual accuracy number honestly rather
  than assuming these fixes alone guarantee a "good" result.

Both are why Phase 1 validates every path explicitly and Phase 2 tracks validation CER
per epoch instead of trusting whichever epoch happens to run last.

## 3.2 Profiling Report

Reports the `data_time_total` / `compute_time_total` numbers Phase 2's training loop
collected. On CPU or a small dataset these numbers won't mean much; the point is having
them automatically on a real GPU run, instead of guessing whether the DataLoader or the
model forward/backward pass is the bottleneck.

In [ ]:
_wall = data_time_total + compute_time_total
print("=== Phase 3 profiling report ===")
print(f"Total data-loading time:  {data_time_total:.1f}s")
print(f"Total compute time:       {compute_time_total:.1f}s")
if _wall > 0:
    print(f"Data-loading share:       {100 * data_time_total / _wall:.0f}%")
    print(f"Overall throughput:       {len(train_dataset) * (NUM_EPOCHS - start_epoch) / _wall:.1f} samples/sec")
    if data_time_total > compute_time_total:
        print("\nData loading dominated compute this run -- since Phase 1 already caches every "
              "tensor in memory, the next lever would be num_workers in the DataLoader cell "
              "(Phase 2, hyperparameters) or confirming pin_memory is actually helping on this GPU.")
    else:
        print("\nCompute dominated data loading -- expected, given Phase 1's caching. GPU "
              "compute is the real ceiling now; mixed precision (already on if USE_AMP) and "
              "batch size (probed below) are the two levers left.")
else:
    print("No timing collected -- Phase 2's training loop didn't run any new epochs this "
          "session (fully resumed from checkpoint, or NUM_EPOCHS already satisfied).")

## 3.3 Empirical Batch-Size Probe

Phase 2 used `BATCH_SIZE = 8` as a reasoned default (TrOCR-base at 384x384 comfortably
fits a bigger batch than the handout's 4 on a T4, especially with mixed precision). This
cell checks that empirically instead of guessing further: try increasingly large batches
against the actual loaded model until one runs out of memory, back off to the last one
that worked. Only meaningful on a real GPU -- skipped on CPU, where there's no OOM signal
to test against. This doesn't change the training that already ran; it's a measured
starting point for your next run if you want to push batch size further.

In [ ]:
def find_max_batch_size(model, dataset, device, start=4, max_candidate=64):
    if device != "cuda":
        print("Not running on GPU -- skipping the empirical probe (no OOM signal to test "
              "against on CPU). BATCH_SIZE above was a reasoned default; this cell is what "
              "would replace that guess with a measured number on your actual T4.")
        return None

    bs, working = start, None
    while bs <= max_candidate:
        try:
            torch.cuda.empty_cache()
            probe_loader = DataLoader(dataset, batch_size=bs, shuffle=True)
            batch = next(iter(probe_loader))
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(pixel_values=pixel_values, labels=labels)
                out.loss.backward()
            model.zero_grad(set_to_none=True)
            working = bs
            print(f"  batch_size={bs}: OK")
            bs *= 2
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f"  batch_size={bs}: out of memory -- stopping here")
            break

    if working:
        print(f"\nLargest batch size that fit: {working}. Current BATCH_SIZE={BATCH_SIZE}.")
        if working > BATCH_SIZE * 2:
            print("There's real headroom -- worth raising BATCH_SIZE (and re-checking the LR "
                  "sanity check below, since a bigger batch usually wants a somewhat higher LR) "
                  "on your next training run.")
    model.zero_grad(set_to_none=True)
    return working


max_batch_size = find_max_batch_size(model, train_dataset, device)

## 3.4 Learning-Rate Sanity Check

Not a full LR-range-test (that needs hundreds of steps) -- a cheap check that
`LEARNING_RATE = 5e-5` isn't badly mis-set. Snapshots the model's current weights, probes
a handful of steps at each candidate LR on the training set, restores the original
weights after each probe (so candidates don't contaminate each other or leave the
already-trained model altered), and reports which candidate dropped loss fastest.

In [ ]:
import copy


def lr_sanity_check(model, loader, candidate_lrs=(1e-5, 5e-5, 1e-4, 3e-4), probe_steps=15):
    original_state = copy.deepcopy(model.state_dict())
    results = {}
    it_template = loader

    for lr in candidate_lrs:
        model.load_state_dict(original_state)
        model.train()
        opt = AdamW(model.parameters(), lr=lr)
        losses = []
        it = iter(it_template)
        for _ in range(probe_steps):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(it_template)
                batch = next(it)
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(pixel_values=pixel_values, labels=labels)
            opt.zero_grad()
            out.loss.backward()
            opt.step()
            losses.append(out.loss.item())
        results[lr] = {"start": losses[0], "end": losses[-1], "drop": losses[0] - losses[-1]}
        print(f"  lr={lr:.0e}: loss {losses[0]:.3f} -> {losses[-1]:.3f}  (drop: {results[lr]['drop']:+.3f})")

    model.load_state_dict(original_state)  # restore -- critical, don't leave probed weights active
    model.zero_grad(set_to_none=True)
    best_lr = max(results, key=lambda k: results[k]["drop"])
    print(f"\nSteepest loss drop over {probe_steps} steps: lr={best_lr:.0e}. "
          f"Current LEARNING_RATE={LEARNING_RATE:.0e}.")
    if best_lr != LEARNING_RATE:
        print("Different from the current setting -- worth trying on a full run, but this is "
              f"only a {probe_steps}-step probe; treat it as a lead, not a verdict.")
    return results


lr_probe_results = lr_sanity_check(model, train_loader)

## 3.5 Best-Checkpoint Selection Summary

Confirms what Phase 2's per-epoch validation actually picked, rather than assuming the
final epoch was the best one.

In [ ]:
print("=== Best-checkpoint selection summary ===")
print(f"Final epoch trained:        {epoch + 1} / {NUM_EPOCHS}")
print(f"Final epoch avg loss:       {epoch_avg_losses[-1]:.4f}" if epoch_avg_losses else "No epochs run this session.")
print(f"Best validation CER seen:   {best_val_cer:.3f}" if best_val_cer < float("inf") else "No validation run yet.")
print(f"Best checkpoint saved to:   {BEST_MODEL_DIR}")
print(f"Latest (resume) checkpoint: {CHECKPOINT_DIR}")
if os.path.isdir(BEST_MODEL_DIR) and os.listdir(BEST_MODEL_DIR):
    print("\nEvaluation above already loaded from best_model/ rather than whatever the last "
          "epoch happened to be -- this is just confirming that happened, not re-doing it.")
else:
    print("\nNo best_model/ checkpoint exists yet -- either validation hasn't run (check "
          "EVAL_EVERY in Phase 2) or this session hasn't trained any epochs.")

---
## Step 4: Save Your Model Locally

This saves the finished model to the cloned repo's data folder on the Colab VM's local disk, alongside the loss history and evaluation metrics. **This alone will NOT survive a Colab runtime disconnect or restart** — it's just staging before Step 5 pushes it somewhere durable.

In [ ]:
import json

# Saved next to the cloned data on the Colab VM's local disk. This is ephemeral --
# it disappears when the runtime recycles -- Step 5 pushes the real durable copy to the Hub.
SAVE_DIR = os.path.join(DATA_DIR, "model")
os.makedirs(SAVE_DIR, exist_ok=True)

try:
    model.save_pretrained(SAVE_DIR)
    processor.save_pretrained(SAVE_DIR)

    metrics = {
        "accuracy_pct": accuracy,
        "cer": overall_cer,
        "wer": overall_wer,
        "loss_first_batch": loss_history[0],
        "loss_last_batch": loss_history[-1],
        "epoch_avg_losses": epoch_avg_losses,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
    }
    with open(os.path.join(SAVE_DIR, "week4_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print(f"Model saved locally to: {SAVE_DIR}")
    print("This is on the Colab VM's disk only -- run Step 5 next to push it to the HF Hub")
    print("before this runtime disconnects, or you'll have to retrain.")
except OSError as e:
    raise RuntimeError(f"Couldn't save to {SAVE_DIR}: {e}. Check you have disk space and write permission.") from e

## Step 5: Push the Model to the Hugging Face Hub

*(Not in the handout.)* This is what actually makes the trained model durable past this Colab session. `push_to_hub()` uploads the model weights, processor config, and a generated model card to a repo on the Hub -- Week 5 (or anyone) can then reload it with `from_pretrained("<your-username>/<repo-name>")` instead of retraining from scratch.

**Fill in `HF_REPO_ID` below** (e.g. `your-hf-username/trocr-urdu-si26-unified`). If the repo doesn't exist yet, `push_to_hub` creates it automatically (as a public repo by default -- pass `private=True` in the `create_repo` call below if you'd rather it not be public).

In [ ]:
from huggingface_hub import HfApi, create_repo

# --- Fill this in ---
HF_REPO_ID = "hamnaheh/trocr-urdu-si26-unified"

create_repo(HF_REPO_ID, exist_ok=True, private=False)  # set private=True to keep it unlisted

commit_message = (
    f"Unified pipeline fine-tune: acc={accuracy:.1f}%, CER={overall_cer:.3f}, WER={overall_wer:.3f}, "
    f"{NUM_EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}"
)

model.push_to_hub(HF_REPO_ID, commit_message=commit_message)
processor.push_to_hub(HF_REPO_ID, commit_message=commit_message)

# Also upload the metrics file saved in Step 4 so it lives alongside the model on the Hub.
api = HfApi()
api.upload_file(
    path_or_fileobj=os.path.join(SAVE_DIR, "week4_metrics.json"),
    path_in_repo="week4_metrics.json",
    repo_id=HF_REPO_ID,
    commit_message="Add Week 4 evaluation metrics",
)

print(f"Pushed to: https://huggingface.co/{HF_REPO_ID}")
print("Reload next week with:")
print(f'  model = VisionEncoderDecoderModel.from_pretrained("{HF_REPO_ID}")')
print(f'  processor = TrOCRProcessor.from_pretrained("{HF_REPO_ID}")')

After running Step 5, check the printed `huggingface.co/...` link to confirm the model repo exists and has files in it, before ending this Colab session.

## Saving This Notebook Back to GitHub

The model itself already lives on the Hugging Face Hub (Step 5) — don't try to `git push` it or the cloned data folder back to GitHub; both are large/binary and git handles that poorly (and the data folder is redundant with what's already in your repo anyway).

For the **notebook file only**:

1. In Colab, go to **File → Save a copy in GitHub**.
2. Pick your repo (`Urdu-OCR-Project-Code-Saviours-SI-26-Humna-Imran` or whichever you cloned in Step 0) and branch.
3. Optionally rename the file (e.g. `SI26_Unified_Pipeline_humna_colab.ipynb`) and add a commit message.
4. Click **OK** — this commits just the `.ipynb` (with its outputs) to your repo. It's a one-click action in the Colab UI, not something to script.

That's the artifact you submit as the "GitHub link to this notebook" in the checklist below.

After running this cell, check that the `model/` folder now exists under your data directory (`ls` it, or use the file explorer) before ending your Codespace session.

## Summary & Submission Checklist

- **GitHub link to this notebook** (data expansion + training + evaluation + audit, all
  in one file) — push this to your repo.
- **`My model accuracy is X%`** — printed in Phase 2's evaluation cell
  (`Accuracy: X% (n/m correct)`).
- **`Training loss went from X to X`** — printed at the end of Phase 2's training loop,
  along with total data-loading/compute time (also in Phase 3's profiling report).
- **Screenshot of training output showing loss decreasing** — Phase 2's loss chart
  (`training_loss.png`) covers this directly.
- **3–5 wrong examples** — auto-generated right after Phase 2's evaluation cell.
- **What changed in the audit** — Phase 3's recap + profiling report + batch-size/LR
  checks, if that context is useful for writeup.

CER/WER, the prediction grid, and `metrics.json` aren't required by any handout, but are
there if useful.